# Consistency: RFT on Countdown with the label-free judge (Llama-3.2-3B)

Rejection-sampling fine-tuning (RFT / expert iteration) as the alternative to GRPO, with everything else as in
`Consistency_GRPO_Countdown34`: the mixed SFT start, the corrupt judge (`DPO_RUN`, sum score, calibration file),
the 3-/4-number training mix, the held-out curve puzzles, pass@16 puzzles, 4- and 3-number test sets and the report table.

Each round ([rft]): sample `RFT_PUZZLES` training puzzles x 16 at T 0.9 with the current policy, score each
completion with the GRPO reward (format + judge; `SELECT = "checker"`: format + checker, the supervised baseline),
keep per puzzle the top `RFT_KEEP` finished completions above the group mean, SFT on them, evaluate.
4 rounds x 300 puzzles x 16 = 19,200 rollouts, the GRPO runs' budget.

Run order: [config] -> [drive + HF] -> [install] -> [vLLM setup] -> [common] -> [model] -> [rewards] -> [vLLM sync fix]
-> [eval] -> [build] -> [pool] -> [held-out] -> [reward calibration] (cached) -> [early stop] (for the curve set) -> [rft].
[rft] resumes after the last finished round (its adapter and optimizer are saved in `LOG_DIR/round<N>/`).


In [ ]:
# [config] Everything you choose for a run lives here. Changing the policy adapter needs a session restart.
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
# Starting policy: a Countdown SFT adapter of Consistency_Countdown. Judge (JUDGE):
#   "base"  the frozen base model (adapter disabled); GRPO continues the SFT LoRA. Runs a-c (2026-10-16) used this with
#           SFT_RUN = "Llama-3.2-3B-Instruct-countdown4000-ep2-lr0.0001-seed0-tr2" (test accuracy 0.196, pass@16 0.665;
#           base judge win 0.82 mean / 0.96 worst)
#   "dpo"   the [judge DPO] adapter of SFT_RUN, a second frozen adapter on the base ([model]). On the gold-SFT samples:
#           win 0.980 mean (0.983 length-matched; base 0.783), real gap completions 0.955 (base 0.541), best-of-16 0.620
#           of pass@16 0.640. It reads the "Answer:" line a lot: steps only 0.856.
#   POLICY_START (JUDGE "dpo"): "sft" = the policy starts at the SFT model (val greedy accuracy 0.22, truncated 0.38);
#           "judge" = at the DPO model (0.10, truncated 0.81: DPO hurt the policy)
#SFT_RUN = "Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp38x38-judge500-mix3x500"
#SFT_RUN = "Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500"
# ^ trace v4 without unsolvable traces ("v5", 2026-10-26): 500 + 500 traces + 500 gold
SFT_RUN = "Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000"   # the mixed SFT (paper runs)
# ^ trace v4 (recomputing check, caught claims, honest stops), 500 + 500 traces + 76 unsolvable + 500 gold (2026-10-24)
#SFT_RUN = "Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000"   # 3000 + 3000 (3-number) tr2 traces + 3000 gold, 1 epoch (runs up to 2026-10-24)
#SFT_RUN = "Llama-3.2-3B-Instruct-countdown4000-ep1-lr0.0001-seed0-tr2-judge2000"   # 4-number only: 4000 tr2 traces + 2000 gold (runs up to 2026-10-21)
JUDGE = "dpo"
POLICY_START = "sft"
# The label-free judge from corrupted reasoning (Consistency_Corrupt_Judge, TRAIN_KINDS empty + cut), under SFT_RUN:
#DPO_RUN = "judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1"

#DPO_RUN = "judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1" #first corrupt judge (swaps, empty and cuts)
# judges of the 4-number-only SFT (2026-10-16..21): judge_dpo_swapslip_n3000_..., judge_dpo_swapslipmiss_n3000_n3x1500_...,
# judge_dpo_swap40slip30miss30_n3000_n3x1500_... (+ _real1000x500: 23,040 checker labels)
DPO_RUN = "judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1"
JUDGE_TAG = ("" if DPO_RUN == "judge_dpo_swapslip_n3000_b0.1_lr2e-05_ep1"      # in calibration and run names
             else "-corrupt_" + DPO_RUN.split("/dpo_")[1].split("_b")[0] if DPO_RUN.startswith("judge_corrupt_")
             else "-" + DPO_RUN.removeprefix("judge_dpo_").split("_b")[0])

# Arm (reward), one per run:
#   "format_only"  the tag rewards: xmlcount + soft/strict format (FORMAT_FUNCS in [rewards])
#   "mean"         format + judge: per-token mean log-prob of solution + "Answer: ..." + end of turn (as on GSM8K)
#   "sum"          format + judge: summed log-prob of the same text (rewards shorter solutions; with MASK_GRAD_BLOCKS the
#                  policy can't shorten the solution directly)
#   "ratio"        format + judge: the sum minus the summed log-prob of the same text under the judge's DPO reference (the
#                  SFT adapter, ADAPTER_PATH, frozen adapter "judge_ref"): J_ratio of judge_reward_grpo.tex, the DPO
#                  implicit reward / beta. JUDGE "dpo" only
#   "mean_worst"   format + judge: z(mean) + z(worst result), worst = the least likely step result (tokens after "=")
#   "pass"         format + judge as a pass/fail verifier: 1 if the judge's mean > the threshold, else 0 (missing = 0).
#                  All wrong answers score alike, so the judge's ranking among wrong answers (which favored gap
#                  solutions: run grpo-mean-c, 2026-10-16) no longer steers the policy
#   "correctness"  format + the exact checker (2.0 if right): the verifiable-reward ceiling
#   "intuitor"     Intuitor (Zhao et al., ICLR 2026): the policy's own self-certainty over its whole completion (mean over
#                  tokens of KL(uniform || p), computed online with the current policy) as the ONLY reward; format, judge
#                  and correctness are logged with weight 0. INTUITOR_RECIPE applies their MATH 1.5B/3B settings, LoRA-
#                  translated (below)
# Every arm logs the judge scores, correctness and the Countdown monitors with weight 0.
ARM = "sum"                 # with JUDGE "dpo": mean (dpo worst is weaker: 0.948, real gaps 0.759)
SC_WEIGHT = 1.0                 # ARM "intuitor": weight of self-certainty. With SCALE_REWARDS False it sets the advantage
                                # scale: [self-certainty probe] prints the weight that matches correctness's
RUN_TAG = "rft1-cut20"                # RFT runs (2026-11-09)
# ---- RFT (this notebook: rejection-sampling fine-tuning instead of GRPO; [rft]). Each of RFT_ROUNDS rounds takes the
# next RFT_PUZZLES training puzzles ([held-out]'s training set in a seeded order) x NUM_GENERATIONS samples at
# TEMPERATURE with the current policy, scores each completion with the GRPO reward of the selection signal:
#   SELECT "judge"    format rewards + the calibrated judge (ARM's plausibility value x its coef): label-free
#   SELECT "checker"  format rewards + the checker's correctness (2.0 if right): the supervised RFT baseline
# and per puzzle keeps the top RFT_KEEP FINISHED completions whose score is above the group mean (GRPO's positive
# advantages). Then SFT on them: RFT_EPOCHS epoch(s), loss on the completion tokens (per-token mean over the batch),
# policy LoRA only, AdamW at RFT_LR (constant), batches of RFT_BATCH (micro-batches of RFT_MICRO), grad clip
# RFT_MAX_GRAD_NORM. No KL term: the few rounds and the selection keep it near the start. Budget: 4 x 300 x 16 = 19,200
# rollouts = the GRPO runs' 150 steps x 128. After each round: the full eval (report table) and the round's adapter.
SELECT = "judge"
RFT_ROUNDS, RFT_PUZZLES, RFT_KEEP = 8, 300, 2     # 8 rounds = 38,400 rollouts (2026-11-11; the first run: 4, then 10)
RFT_LR, RFT_EPOCHS, RFT_BATCH, RFT_MICRO, RFT_MAX_GRAD_NORM = 2e-5, 1, 16, 4, 1.0
# RFT_SKIP_Q: after the per-puzzle selection, drop the selected completions scored below this quantile of the round's
# selected scores (label-free: the selection signal's own least confident picks); 0 keeps all. On the first run's pools
# (r4x300, rounds 1-6) 0.2 keeps ~480 of 600 and raises the selected correct share from 0.77-0.81 to 0.90-0.95.
RFT_SKIP_Q = 0.2
EXPRESSION_REWARD = True        # also reward "the answer is a whole-number expression" (check_answer's form)
EXPRESSION_ACCEPTS_STOP = False  # (2026-10-25) the expression reward also pays a stop answer ("none", the v4 SFT's format):
                                # with False every honest stop lost 0.5 to any guessed expression. False = the runs up to 2026-10-25
#MASK_GRAD_BLOCKS = ("solution", "answer")   # no policy gradient on the text of these blocks (the last of each; tags
                                 # kept): still generated, read as context, judged and graded, but the reward can only
                                 # shape the reasoning (and the tags). () = no masking
MASK_GRAD_BLOCKS = ()
# Judge settings. PLAUSIBILITY_MAX_SOLUTION_TOKENS: longer solutions score as missing (Countdown solutions: 3 lines).
PLAUSIBILITY_MAX_SOLUTION_TOKENS = 128
# JUDGE_NEEDS_ANSWER: a completion without an <answer> scores as missing (else its steps alone are judged). The DPO
# judge's verdict leans on the answer line: on the gold-SFT samples, steps only it ranks correct over real gap solutions
# just 0.569 (with the answer 0.952), so "gap steps, no answer" would be its weakest spot (judge probe v2, 2026-10-16).
JUDGE_NEEDS_ANSWER = True
# MISSING_AT (arms mean / sum / mean_worst): the value of a completion the judge can't score (no solution, over the cap,
# no answer). "below_floor": just below the 1st percentile, worse than any wrong answer (runs up to grpo-mean-c: strong
# pressure to finish at any cost); "wrong_median": the median value of the scored wrong answers in calibration (an
# unfinished search = a typical wrong answer). The "pass" arm always gives missing 0.
MISSING_AT = "wrong_median"
# "pass" arm: PASS_THRESHOLD = None sets the threshold from calibration so that a share PASS_MAX_FALSE_PASS of the scored
# wrong answers pass (DPO judge at the start: -0.7 passes 0.90 of correct, 0.05 of wrong); a number fixes it.
PASS_THRESHOLD = None
PASS_MAX_FALSE_PASS = 0.05

# Training puzzles: POOL_CANDIDATES TinyZero 4-number puzzles (seeded draw without the 200 test puzzles, the one-shot
# example and all SFT puzzles).
#   USE_POOL = False  train on all of them (minus a random held-out set): the puzzles as they come
#   USE_POOL = True   [pool] samples each PASS_SAMPLES times with the starting adapter first; train only on pass rates in
#                     PASS_BAND (at least one right and one wrong of 8: every group has a correctness signal)
USE_POOL = False
TRAIN_MIX_3 = 0.5               # share of 3-number puzzles in training (Gandhi et al. 2025: 0.5), USE_POOL False only: the
                                # 4-number training puzzles plus a new seeded draw of 3-number ones (none in any eval set)
POOL_CANDIDATES = 4000
PASS_SAMPLES = 8
PASS_BAND = (0.1, 0.75)
POOL_SEED = 0

# Training (as the 8B GSM8K run b, which stayed stable at 5e-5; 1e-4 collapsed).
MAX_COMPLETION_LENGTH = 1024     # SFT completions: median ~280 tokens; loops run to the cap
SMOKE_RUN = False               # True: pipeline test (~10-15 min): 10 steps, 64 candidates, 16 calibration puzzles,
                                 # held-out curve every 5 steps, 50 test puzzles; own cache files (the real run rebuilds)
MAX_STEPS = 150                  # the paper's runs: 150 (the pass continuation: 300)
# POSITIVE_ADV_SCALE: negative-sample reinforcement (Zhu et al. 2025, Weighted-REINFORCE): positive advantages are
# multiplied by this, negative ones kept. Pushing probability onto the sampled correct paths is what narrows the policy
# (run f: distinct first moves per group 13 -> 6); pushing it off wrong paths spreads it over all other outputs.
# 1.0 = plain GRPO
POSITIVE_ADV_SCALE = 1.0
# LABEL_BUDGET (ARM "correctness" only): an equal-label control. MAX_STEPS = LABEL_BUDGET / COMPLETIONS_PER_STEP (rounded
# up), so the run grades about as many completions with the checker as the judge arm consumed: the judge's graded real
# completions (Consistency_Countdown [judge DPO] prints them; DPO_RUN/labels.json) + [reward calibration]'s 128 x
# NUM_GENERATIONS. None = MAX_STEPS as set. [train] saves each run's label counts to LOG_DIR/labels.json
LABEL_BUDGET = None
# TRAIN_PASS_K: Pass@k Training (Chen et al. 2025; countdown/passk.py): the correctness part of each advantage is replaced
# by the analytical pass@k advantage (normalized by the SD of the k-subset rewards, as in the paper). Rarely solved
# puzzles get about the plain push, mastered ones (fewer than k wrong of NUM_GENERATIONS) none. Paper: N 32, k 8, no KL.
# None = plain GRPO. ARM "correctness" with SCALE_REWARDS False only
TRAIN_PASS_K = None
PASS_AT_K = 16                   # sampled pass@k (at PASS_TEMPERATURE) on PASS_N held-out puzzles (half from each
PASS_N = 100                     # curve stratum: band + unscored): before training (cached), at PASS_EVAL_STEPS, and
PASS_EVAL_STEPS = (50, 100)     # the paper's runs: (50, 100); after training; 0 = skip. ~PASS_N x PASS_AT_K completions each time (diversity check)
RICH_EVAL_STEPS = (50,)          # the paper's runs: (50,); also the full "after" evals at these steps ([early stop]): test set, held-out strata, mixed
                                 # 3/4-number test (LOG_DIR/eval_*_step{N}*); () = only at the end
PASS_TEMPERATURE = 0.9           # fixed across runs, whatever TEMPERATURE trains at (the cached "before" is at 0.9)
RUN_TEMP_SWEEP = False           # [temperature sweep] (after [held-out]): pass@k of fixed checkpoints at several temperatures
LEARNING_RATE = 2e-5
# LR_SCHEDULE (transformers names): "cosine" (warmup, then anneal to 0 at MAX_STEPS), "constant_with_warmup"
# (LEARNING_RATE after warmup: new runs from 2026-11-03), or "constant" (LEARNING_RATE from step 1, NO warmup: for
# continuations, so the pressure doesn't decay). The paper's 150-step runs used cosine.
LR_SCHEDULE = "cosine"           # mean-34 / format-34-150steps
# CONTINUE_FROM: (run name, step) of a run in this GRPO_DIR. This run (its own RUN_TAG) starts as an exact continuation
# of that checkpoint: [build] copies it (adapter, optimizer, scheduler, RNG, step) and the curve / metrics rows up to that
# step into LOG_DIR, and [train] resumes from it. Same reward and calibration file; the KL reference stays the starting
# policy. None = start at the SFT adapter.
#CONTINUE_FROM = ("grpo-mean-missmed-mix3_0.5-expr-kl0.02-swap30extra30slip20miss20_n2250_n3x2250-mean-34", 100)
#CONTINUE_FROM = ("grpo-pass-fp0.05-mix3_0.5-expr-kl0.02-corrupt_empty-cut-v3-mix-corrupt-pass", 150)
CONTINUE_FROM = None
# Trainer settings (defaults = runs e-g); INTUITOR_RECIPE overrides them for ARM "intuitor".
TEMPERATURE = 0.9                # sampling temperature for training groups and calibration
KL_BETA = 0.02                   # KL penalty to the starting policy (a frozen copy, adapter "ref"); 0 = off
NUM_GENERATIONS = 16            # completions per puzzle (a group)
COMPLETIONS_PER_STEP = 128       # NUM_GENERATIONS x puzzles per step
WARMUP_STEPS = 20
SCALE_REWARDS = False           # divide advantages by the group's reward SD (GRPO's default; True up to 2026-10-24, as mean-34)
LOSS_TYPE, MAX_GRAD_NORM, EPSILON = "dapo", 0.2, 0.5
ADAM_BETA2, WEIGHT_DECAY = 0.99, 0.0
# INTUITOR_RECIPE (ARM "intuitor"): their Table 10 (MATH 1.5B/3B, full fine-tuning) translated to our LoRA: LR x 10
# (3e-6 -> 3e-5), KL 0.005 (their text; the table's 0.0005 looks like a typo), 7 samples x 128 puzzles per step, 58
# steps, 10% warmup, group-normalized rewards, GRPO loss, AdamW beta2 .999, no weight decay, no filter, plain GRPO.
# Ours kept: max completion (theirs 3072), the prompt. Their reward lives in code/countdown/intuitor.py.
INTUITOR_RECIPE = False
if ARM == "intuitor" and INTUITOR_RECIPE:
    LEARNING_RATE, KL_BETA, NUM_GENERATIONS, COMPLETIONS_PER_STEP, MAX_STEPS, WARMUP_STEPS = 3e-5, 0.005, 7, 896, 58, 6
    SCALE_REWARDS, LOSS_TYPE, MAX_GRAD_NORM, EPSILON, ADAM_BETA2, WEIGHT_DECAY = True, "grpo", 1.0, 0.2, 0.999, 0.0
    USE_POOL, POSITIVE_ADV_SCALE, EXPRESSION_REWARD = False, 1.0, False
VLLM_GPU_MEMORY_UTILIZATION = 0.35
MICRO_BATCH = 16

# The test set and the SFT puzzles, exactly as Consistency_Countdown draws them (same seeds and sizes).
N_TEST, N_SFT, N_SFT_VAL, SEED = 200, 4000, 100, 0

if SMOKE_RUN:
    POOL_CANDIDATES = 64
    RFT_ROUNDS, RFT_PUZZLES = 2, 8
if LABEL_BUDGET:
    assert ARM == "correctness", "LABEL_BUDGET is for the correctness control"
    MAX_STEPS = -(-LABEL_BUDGET // COMPLETIONS_PER_STEP)

DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
ADAPTER_PATH = f"{DRIVE}/countdown/sft/{SFT_RUN}/adapter"
JUDGE_ADAPTER_PATH = f"{DRIVE}/countdown/sft/{SFT_RUN}/{DPO_RUN}/adapter" if JUDGE == "dpo" else None
GRPO_DIR = (f"{DRIVE}/countdown/grpo/{SFT_RUN}"                 # pool, calibration and runs per policy x judge
            + (f"-judge_dpo-start_{POLICY_START}" if JUDGE == "dpo" else ""))
CALIBRATION_PATH = (f"{GRPO_DIR}/calibration_cap{PLAUSIBILITY_MAX_SOLUTION_TOKENS}"   # per puzzle mix: the spreads differ
                    + ("_needans" if JUDGE_NEEDS_ANSWER else "") + ("" if USE_POOL else "_nopool")
                    + (f"_mix3_{TRAIN_MIX_3:g}" if TRAIN_MIX_3 else "") + JUDGE_TAG.replace("-", "_")
                    + ("_smoke" if SMOKE_RUN else "") + ".json")
POOL_FILE = f"{GRPO_DIR}/pass_rates_n{POOL_CANDIDATES}_s{PASS_SAMPLES}_seed{POOL_SEED}_max{MAX_COMPLETION_LENGTH}.csv"
JUDGE_ARMS = ("mean", "sum", "mean_worst", "pass") + (("ratio",) if JUDGE == "dpo" else ())   # the arms that train on the judge
RUN_NAME = (f"grpo-{ARM}" + ("-missmed" if ARM in ("mean", "sum", "mean_worst", "ratio") and MISSING_AT == "wrong_median" else "")
            + (f"-fp{PASS_MAX_FALSE_PASS:g}" if ARM == "pass" and PASS_THRESHOLD is None else "")
            + (f"-t{PASS_THRESHOLD:g}" if ARM == "pass" and PASS_THRESHOLD is not None else "")
            + (f"-mix3_{TRAIN_MIX_3:g}" if TRAIN_MIX_3 else "") + (("-exprstop" if EXPRESSION_ACCEPTS_STOP else "-expr") if EXPRESSION_REWARD else "") + ("-mask_" + "_".join(MASK_GRAD_BLOCKS) if MASK_GRAD_BLOCKS else "")
            + (f"-pos{POSITIVE_ADV_SCALE:g}" if POSITIVE_ADV_SCALE != 1.0 else "") + (f"-pk{TRAIN_PASS_K}" if TRAIN_PASS_K else "")
            + (f"-lab{LABEL_BUDGET}" if LABEL_BUDGET else "") + (f"-kl{KL_BETA:g}" if KL_BETA else "") + (f"-t{TEMPERATURE:g}" if TEMPERATURE != 0.9 else "") + ("-recipe" if ARM == "intuitor" and INTUITOR_RECIPE else "")
            + (JUDGE_TAG if ARM in JUDGE_ARMS else "") + ("-smoke" if SMOKE_RUN else "") + f"-{RUN_TAG}")
RUN_NAME = (f"rft-{SELECT}"                                 # this notebook: the RFT run (GRPO_DIR caches shared)
            + (f"-{ARM}" + ("-missmed" if MISSING_AT == "wrong_median" else "") + JUDGE_TAG if SELECT == "judge" else "")
            + f"-r{RFT_ROUNDS}x{RFT_PUZZLES}k{RFT_KEEP}" + (f"-skip{RFT_SKIP_Q:g}" if RFT_SKIP_Q else "") + f"-lr{RFT_LR:g}" + (f"-ep{RFT_EPOCHS}" if RFT_EPOCHS != 1 else "")
            + (f"-mix3_{TRAIN_MIX_3:g}" if TRAIN_MIX_3 else "") + ("-smoke" if SMOKE_RUN else "") + f"-{RUN_TAG}")
LOG_DIR = f"{GRPO_DIR}/runs/{RUN_NAME}"

assert SELECT in ("judge", "checker") and (SELECT == "checker" or ARM in ("mean", "sum", "ratio", "mean_worst")), \
    "RFT: SELECT 'judge' needs a continuous judge ARM (sum: the theory's score)"
assert ARM in ("format_only", "correctness", "intuitor") + JUDGE_ARMS, f"{ARM} (ratio needs JUDGE 'dpo')"
assert COMPLETIONS_PER_STEP % NUM_GENERATIONS == 0 and COMPLETIONS_PER_STEP % MICRO_BATCH == 0, "batch sizes don't divide"
assert JUDGE in ("base", "dpo") and POLICY_START in ("sft", "judge"), (JUDGE, POLICY_START)
assert MISSING_AT in ("below_floor", "wrong_median"), MISSING_AT
assert not TRAIN_MIX_3 or (not USE_POOL and 0 < TRAIN_MIX_3 < 1), "TRAIN_MIX_3 needs USE_POOL False and 0 < share < 1"
assert not TRAIN_PASS_K or (ARM == "correctness" and not SCALE_REWARDS and 1 <= TRAIN_PASS_K < NUM_GENERATIONS), \
    "TRAIN_PASS_K needs ARM correctness, SCALE_REWARDS False and k < NUM_GENERATIONS"
print(f"MODEL={MODEL}\nadapter={ADAPTER_PATH}\njudge={JUDGE} {JUDGE_ADAPTER_PATH or ''}"
      + (f"  policy starts at {POLICY_START}" if JUDGE == "dpo" else "") + f"\nARM={ARM}  RUN_TAG={RUN_TAG}  SMOKE_RUN={SMOKE_RUN}  "
      f"lr={LEARNING_RATE:g}  steps={MAX_STEPS}\nLOG_DIR={LOG_DIR}")

In [ ]:
# [drive + HF] Mount Drive and log in to HF (Colab Secret HF_TOKEN; Llama is gated). Re-running is harmless.
import os
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # read by transformers and vLLM; no huggingface_hub import before [install]
for f in ("countdown.py", "prompts.py", "traces.py", "rewards.py", "intuitor.py", "passk.py"):
    assert os.path.exists(f"{CODE_DIR}/{f}"), f"{CODE_DIR}/{f} missing (copy countdown/ from the repo)"
assert os.path.exists(f"{ADAPTER_PATH}/adapter_model.safetensors"), f"no adapter at {ADAPTER_PATH}"
assert JUDGE != "dpo" or os.path.exists(f"{JUDGE_ADAPTER_PATH}/adapter_model.safetensors"), \
    f"no DPO adapter at {JUDGE_ADAPTER_PATH} (Consistency_Countdown [judge DPO])"

In [ ]:
# [install] Same packages as Consistency_GRPO, once per runtime: skipped when vllm and trl are installed. No restart:
# nothing they replace (torch, transformers) is imported before this cell. torchao is removed either way: Colab ships
# 0.10, and peft refuses to import below 0.16 (nothing here uses it).
import importlib.util, subprocess, sys
if importlib.util.find_spec("vllm") and importlib.util.find_spec("trl"):
    print("[install] skipped: vllm and trl are installed")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl[vllm]", "peft"], check=True)
    print("[install] done")
if importlib.util.find_spec("torchao"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")

In [ ]:
# [vLLM setup] Run once per session BEFORE anything touches the GPU.
import os, sys
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the vLLM engine in this process; CUDA can't re-init in a forked child
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Colab's stdout/stderr have no OS file descriptor; vLLM asks for one during startup.
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import importlib.metadata as md
for p in ["torch", "transformers", "trl", "peft", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")

In [ ]:
# [common] Tokenizer, the Countdown code (countdown.py, traces.py, rewards.py from CODE_DIR), and the puzzles:
#   test         the N_TEST puzzles of Consistency_Countdown's discrimination test (same draw), for before/after evals
#   SFT puzzles  the adapter's training puzzles (same draw as Consistency_Countdown), excluded from everything here
#   RL draw      a third seeded draw without both: POOL_CANDIDATES candidates (scored by [pool]) + UNSCORED_N more
#                (never scored or trained on: a held-out stratum)
import os, sys, re, json, random, importlib
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
RFT_ROUNDS = 10

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = "<|finetune_right_pad_id|>"   # not <|eot_id|> (the EOS)
tokenizer.chat_template = re.sub(r'strftime_now\([^)]*\)', '"26 Jul 2024"', tokenizer.chat_template)  # pinned date

def chat_prompt_ids(tok, messages) -> list[int]:
    """Templated prompt as token ids, no extra BOS (vLLM would add a second one to text)."""
    return tok(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
               add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)

import prompts, countdown, traces, rewards
for _m in (prompts, countdown, traces, rewards):   # prompts first: the others import from it
    importlib.reload(_m)
from countdown import check_answer, solve
from prompts import countdown_prompt, countdown_question, bare_question
from rewards import tag, completion_text, diagnose

UNSCORED_N = 100
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as_puzzle = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])

# test + one-shot example: Consistency_Countdown's [puzzles] draw
_draw = _tz.select(np.random.default_rng(SEED).choice(len(_tz), size=min(len(_tz), 20 * N_TEST), replace=False))
_four = [_as_puzzle(r) for r in _draw if len(r["nums"]) == 4]
EXAMPLE, test_puzzles = _four[0], _four[1:N_TEST + 1]
_seen = {_key(p) for p in test_puzzles} | {_key(EXAMPLE)}
# the SFT puzzles: Consistency_Countdown's second draw
_sft = []
for r in _tz.select(np.random.default_rng(SEED + 1).choice(len(_tz), size=min(len(_tz), 6 * (N_SFT + N_SFT_VAL)),
                                                           replace=False)):
    p = _as_puzzle(r)
    if len(p["numbers"]) == 4 and _key(p) not in _seen:
        _seen.add(_key(p))
        _sft.append(p)
_sft = _sft[:N_SFT + N_SFT_VAL]
_seen = {_key(p) for p in test_puzzles} | {_key(EXAMPLE)} | {_key(p) for p in _sft}
# RL draw: candidates for [pool], then the unscored held-out puzzles
_rl = []
for r in _tz.select(np.random.default_rng(SEED + 2).choice(len(_tz), size=min(len(_tz), 4 * (POOL_CANDIDATES + UNSCORED_N)),
                                                           replace=False)):
    p = _as_puzzle(r)
    if len(p["numbers"]) == 4 and _key(p) not in _seen:
        _seen.add(_key(p))
        _rl.append(p)
assert len(_rl) >= POOL_CANDIDATES + UNSCORED_N, f"only {len(_rl)} RL puzzles in the draw"

def puzzle_dataset(puzzles, first_id=0):
    """Rows for TRL: the chat prompt plus numbers / target (passed to the reward functions) and a puzzle id."""
    return Dataset.from_list([{"prompt": countdown_prompt(p["numbers"], p["target"]), "numbers": p["numbers"],
                               "target": p["target"], "pid": first_id + i} for i, p in enumerate(puzzles)])

candidates = puzzle_dataset(_rl[:POOL_CANDIDATES])                          # pid 0 .. POOL_CANDIDATES - 1
unscored = puzzle_dataset(_rl[POOL_CANDIDATES:POOL_CANDIDATES + UNSCORED_N], first_id=POOL_CANDIDATES)
test_set = puzzle_dataset(test_puzzles, first_id=10**6)                      # pids kept apart from the RL draw
# 3-number test puzzles from the same draw (none are in SFT or RL, which are 4-number only): with test_set, the 50/50
# 3- and 4-number mix Gandhi et al. 2025 train and evaluate on ([mixed test])
test3_set = puzzle_dataset([p for p in (_as_puzzle(r) for r in _draw if len(r["nums"]) == 3)
                            if _key(p) not in _seen][:N_TEST], first_id=2 * 10**6)
# 5-number test puzzles: generated (TinyZero has 3 and 4 numbers only; none in any SFT, judge or GRPO data), the same
# draw as Consistency_Countdown's [puzzles] (seed SEED + 5, solvable by construction, target 10-100): its 5-number judge
# eval used these first N_TEST5 ([5-number test])
N_TEST5 = 100
_r5 = random.Random(SEED + 5)
test5_set = puzzle_dataset([{"numbers": g["numbers"], "target": g["target"]}
                            for g in (countdown.generate(_r5, n_numbers=5, count_solutions=False) for _ in range(N_TEST5))],
                           first_id=3 * 10**6)
# TRAIN_MIX_3: 3-number training puzzles, a fourth seeded draw without every puzzle above and every 3-number puzzle of
# the test draw (test3_set and the ones after it); as many as make TRAIN_MIX_3 of the training set ([held-out] adds them)
train3 = None
if TRAIN_MIX_3:
    _n3 = round(TRAIN_MIX_3 / (1 - TRAIN_MIX_3) * (POOL_CANDIDATES - min(100, POOL_CANDIDATES // 5)))
    _no3 = (_seen | {_key(_as_puzzle(r)) for r in _draw}       # not the test draw, nor any puzzle of the SFT draw (the
            | {_key(_as_puzzle(r)) for r in _tz.select(np.random.default_rng(SEED + 1).choice(   # judge's 3-number
                len(_tz), size=min(len(_tz), 6 * (N_SFT + N_SFT_VAL)), replace=False))})       # DPO pairs come from it)
    _p3 = []
    for r in _tz.select(np.random.default_rng(SEED + 3).choice(len(_tz), size=min(len(_tz), 4 * _n3), replace=False)):
        p = _as_puzzle(r)
        if len(p["numbers"]) == 3 and _key(p) not in _no3:
            _no3.add(_key(p))
            _p3.append(p)
    assert len(_p3) >= _n3, f"only {len(_p3)} 3-number puzzles in the draw"
    train3 = puzzle_dataset(_p3[:_n3], first_id=5 * 10**6)
print(f"test {len(test_set)} (as in Consistency_Countdown) + {len(test3_set)} 3-number + {len(test5_set)} 5-number, SFT puzzles excluded {len(_sft)}, "
      f"candidates {len(candidates)}, unscored held-out {len(unscored)}")
print("example prompt:", candidates[0]["prompt"][0]["content"][:160], "...")

In [ ]:
# [model] The policy is a LoRA on the base model (adapter "default", trainable; GRPO continues it). JUDGE (in [config]):
#   "base"  the judge is the base model: model.disable_adapter() (no extra memory).
#   "dpo"   the judge is the base model + the [judge DPO] adapter, loaded as a second, frozen adapter "judge" (the "dpo"
#           judge of the discrimination test exactly); judging switches to it under no_grad and back (judge_weights in
#           [rewards]). POLICY_START "sft": "default" = the SFT adapter; "judge": "default" starts as a copy of the DPO
#           adapter. No merging: both are single adapters on the bf16 base, as in runs a-c; with KL_BETA a third,
#           frozen adapter "ref" (a copy of the starting policy) is the KL reference. (The first version merged
#           the DPO adapter and stacked SFT - DPO as a rank-32 policy LoRA: the trainer's and vLLM's log-probs then
#           differed 4x more than in runs a-c, importance-sampling ratio ~0.4 from step 1; runs grpo-mean-c/d.)
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
_policy_init = JUDGE_ADAPTER_PATH if (JUDGE == "dpo" and POLICY_START == "judge") else ADAPTER_PATH
model = PeftModel.from_pretrained(model, _policy_init, is_trainable=True)          # adapter "default"
_cfg = model.peft_config["default"]
print(f"policy adapter (default) from {_policy_init}: r={_cfg.r} alpha={_cfg.lora_alpha} targets={sorted(_cfg.target_modules)}")
if JUDGE == "dpo":
    model.load_adapter(JUDGE_ADAPTER_PATH, adapter_name="judge", is_trainable=False)
    # the judge's DPO reference (Consistency_Countdown [judge DPO]: reference = the SFT adapter), for the "ratio" score
    model.load_adapter(ADAPTER_PATH, adapter_name="judge_ref", is_trainable=False)
    model.set_adapter("default")                     # active = the policy; PEFT marks only "default" trainable
    print(f"judge adapter (frozen) from {JUDGE_ADAPTER_PATH}; its DPO reference (frozen, judge_ref) from {ADAPTER_PATH}")
else:
    print("judge = base model (adapter disabled)")
if KL_BETA:   # TRL uses an adapter named "ref" as the KL reference: a frozen copy of the starting policy
    print(f"KL reference: TRL copies the starting policy adapter to 'ref' in [build], beta {KL_BETA:g}")


model.print_trainable_parameters()
assert all("lora_" in n for n, p in model.named_parameters() if p.requires_grad), "non-LoRA parameters are trainable"
assert any(p.requires_grad for n, p in model.named_parameters() if "lora_" in n), "adapter loaded frozen"
assert not any(p.requires_grad for n, p in model.named_parameters() if ".judge." in n or ".ref." in n or ".judge_ref." in n), \
    "judge, judge_ref or ref adapter is trainable"

In [ ]:
# [rewards] Format rewards (xmlcount, soft and strict format: verbatim from Consistency_GRPO), the Countdown rewards
# and monitors (rewards.py), the judge, and the step monitor (printed every step, completions dumped every 20 steps).
# The judge = the base model (policy adapter disabled), or with JUDGE "dpo" the base + the frozen DPO adapter ("judge")
# scores the <solution> text + "\nAnswer: <answer>" + end of turn, after a neutral system prompt + the bare puzzle, as
# the GSM8K reward does (solutions over PLAUSIBILITY_MAX_SOLUTION_TOKENS, and with JUDGE_NEEDS_ANSWER completions without
# an answer, score as missing). Three raw scores from one forward pass:
#   mean   per-token mean log-prob
#   sum    summed log-prob
#   worst  the least likely step result: summed log-prob of the tokens after "=" on each line, lowest over the lines
#          (NaN if no line has "=")
# and with JUDGE "dpo" a second pass under the judge's DPO reference ("judge_ref"): ref_sum, the summed log-prob of the
# same text; ratio = sum - ref_sum.
# The trained plausibility value, per ARM, from [reward calibration] (CAL; floors clip outliers; missing = the MISSING_AT
# level: just below the floor, or the median wrong answer):
#   mean        max(mean, floor)
#   sum         max(sum, floor)
#   ratio       sum - ref_sum, clipped to [floor, ceiling] (1st / 99th percentile: a log-ratio has no natural top)
#   mean_worst  z(max(mean, floor_mean)) + z(max(worst, floor_worst)) with the calibration's means / SDs (worst missing:
#               its floor z minus a margin)
#   pass        1 if mean > the calibrated threshold, else 0 (missing 0)
# The other arms log the "mean" value with weight 0. Before calibration the raw mean is logged (weight 0 everywhere).
# ARM "intuitor": self_certainty (intuitor.make_self_certainty) is the only trained reward; everything else is logged.
import math, inspect, contextlib
from trl.trainer.utils import selective_log_softmax
import rewards as R
import intuitor                  # ARM "intuitor": its reward, make_self_certainty (code/countdown/intuitor.py)
importlib.reload(intuitor)

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>") == 1:
        count += 0.125
    if text.count("</reasoning>") == 1:
        count += 0.125
    if text.count("<solution>") == 1:
        count += 0.005
    if text.count("</solution>") == 1:
        count += 0.005
    if text.count("<answer>") == 1:
        count += 0.125
    if text.count("</answer>") == 1:
        count += 0.125
    if "</answer>" in text:
        # Everything after the FIRST </answer> is penalized, so repeating the tag can't switch the penalty off.
        count -= (len(text.split("</answer>", 1)[1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    return [count_xml(completion_text(c)) for c in completions]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>[\s\S]*?</reasoning>\s*<solution>[\s\S]*?</solution>\s*<answer>[\s\S]*?</answer>"
    return [0.5 if re.match(pattern, completion_text(c)) else 0.0 for c in completions]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>\s*.*?\s*</reasoning>\s*<solution>\s*.*?\s*</solution>\s*<answer>\s*.*?\s*</answer>"
    return [0.5 if re.search(pattern, completion_text(c), re.DOTALL) else 0.0 for c in completions]

FORMAT_FUNCS = ([xmlcount_reward_func, soft_format_reward_func, strict_format_reward_func]
                + ([R.expression_or_stop_reward_func if EXPRESSION_ACCEPTS_STOP else R.expression_reward_func]
                   if EXPRESSION_REWARD else []))   # 0.5 if the answer is a bare expression (or, with EXPRESSION_ACCEPTS_STOP, a stop)

# ---- judge
NEUTRAL_SYSTEM_PROMPT = "Please solve the following problem step-by-step, making sure that each step is well-justified."

def result_groups(text, enc):
    """Per line containing "=": the indices of the tokens of its result (from just after the last "=" to the line end)."""
    groups, pos = [], 0
    for line in text.split("\n"):
        if "=" in line:
            s, e = pos + line.rindex("=") + 1, pos + len(line)
            g = [k for k, (a, b) in enumerate(enc["offset_mapping"]) if a < e and b > s]
            if g:
                groups.append(g)
        pos += len(line) + 1
    return groups

@contextlib.contextmanager
def judge_weights(model, adapter="judge"):
    """The judge's weights for the duration: the base model (adapter disabled) or, with JUDGE "dpo", the frozen
    `adapter` ("judge", or its DPO reference "judge_ref"); then back to the policy adapter, which alone is trainable."""
    if JUDGE != "dpo":
        with model.disable_adapter():
            yield
        return
    model.set_adapter(adapter)
    try:
        yield
    finally:
        model.set_adapter("default")
        for n, p in model.named_parameters():        # set_adapter makes the active adapter trainable: only the policy is
            if "lora_" in n and ".default." not in n:
                p.requires_grad_(False)

def judge_raw(owner, prompts, completions, chunk_size=8):
    """(mean, worst, sum, ref_sum) per completion as float arrays, NaN where there is nothing to score (no solution,
    over the cap, or no answer with JUDGE_NEEDS_ANSWER). Scored under owner.model with the judge's weights
    (judge_weights); ref_sum under the judge's DPO reference "judge_ref" (all NaN without JUDGE "dpo")."""
    tok, model = owner.processing_class, owner.model
    fwd = getattr(model, "_original_forward", model.forward)   # skip accelerate's autocast + fp32 logits copy
    means, worsts, sums, ref_sums = (np.full(len(completions), np.nan) for _ in range(4))
    items = []
    for i, (prompt, c) in enumerate(zip(prompts, completions)):
        text = completion_text(c)
        sol, ans = tag(text, "solution"), tag(text, "answer")
        if not sol or len(tok(sol, add_special_tokens=False)["input_ids"]) > PLAUSIBILITY_MAX_SOLUTION_TOKENS:
            continue
        if JUDGE_NEEDS_ANSWER and not ans:
            continue                                            # missing, like no solution
        target = sol + (f"\nAnswer: {ans}" if ans else "")
        enc = tok(target, add_special_tokens=False, return_offsets_mapping=True)
        p_ids = chat_prompt_ids(tok, [{"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
                                      {"role": "user", "content": bare_question(prompt)}])
        items.append((i, p_ids, enc["input_ids"] + [tok.eos_token_id], result_groups(target, enc)))
    for adapter in (("judge", "judge_ref") if JUDGE == "dpo" else ("judge",)):
        with torch.no_grad(), judge_weights(model, adapter):
            for start in range(0, len(items), chunk_size):
                chunk = items[start:start + chunk_size]
                L = max(len(p) + len(t) for _, p, t, _ in chunk)
                ids = torch.full((len(chunk), L), tok.pad_token_id, dtype=torch.long)
                attn = torch.zeros_like(ids)
                for j, (_, p, t, _) in enumerate(chunk):
                    ids[j, :len(p) + len(t)] = torch.tensor(p + t)
                    attn[j, :len(p) + len(t)] = 1
                ids, attn = ids.to(model.device), attn.to(model.device)
                per_pos = selective_log_softmax(fwd(input_ids=ids, attention_mask=attn).logits[:, :-1], ids[:, 1:]).float()
                for j, (i, p, t, groups) in enumerate(chunk):
                    lps = per_pos[j, len(p) - 1:len(p) + len(t) - 1].cpu().numpy()   # positions predicting t
                    if adapter == "judge_ref":
                        ref_sums[i] = lps.sum()
                        continue
                    means[i], sums[i] = lps.mean(), lps.sum()
                    worsts[i] = min(lps[g].sum() for g in groups) if groups else np.nan
                del per_pos
    return means, worsts, sums, ref_sums

CAL = json.load(open(CALIBRATION_PATH)) if os.path.exists(CALIBRATION_PATH) else None

def plausibility_value(m, w, sm, arm, rs=None):
    """The trained plausibility value from raw mean / worst / sum / ref_sum (arrays); before calibration: raw mean, NaN -> None.
    Missing (NaN) gets the calibration's MISSING_AT level; the "pass" arm gives 1 above the threshold, else 0."""
    if CAL is None:
        return [None if math.isnan(x) else float(x) for x in m]
    if arm == "pass":
        thr = CAL["pass"]["threshold"]
        return [0.0 if math.isnan(x) else float(x > thr) for x in m]
    miss = lambda c: c.get(f"missing_{MISSING_AT}", c.get("missing"))   # "missing": files from before MISSING_AT
    if arm == "sum":
        c = CAL["sum"]
        return [miss(c) if math.isnan(x) else float(max(x, c["floor"])) for x in sm]
    if arm == "ratio":
        c = CAL["ratio"]
        return [miss(c) if math.isnan(x) else float(min(max(x, c["floor"]), c["ceiling"])) for x in sm - rs]
    if arm == "mean_worst":
        c = CAL["mean_worst"]
        zm = (np.maximum(m, c["floor_mean"]) - c["mu_mean"]) / c["sd_mean"]
        zw = np.where(np.isnan(w), c["missing_z_worst"], (np.maximum(w, c["floor_worst"]) - c["mu_worst"]) / c["sd_worst"])
        v = zm + zw
        return [miss(c) if math.isnan(x) else float(x) for x in v]
    c = CAL["mean"]
    return [miss(c) if math.isnan(x) else float(max(x, c["floor"])) for x in m]

_LAST_JUDGE = {}   # this batch's raw scores, for the monitors below (matched by the identity of `completions`)

def make_plausibility(owner):
    arm = ARM if ARM in JUDGE_ARMS else "mean"
    def plausibility(prompts, completions, **kwargs):
        m, w, sm, rs = judge_raw(owner, prompts, completions)
        _LAST_JUDGE.update(id=id(completions), mean=m, worst=w, sum=sm, ratio=sm - rs)
        return plausibility_value(m, w, sm, arm, rs)
    plausibility.__name__ = "plausibility"
    return plausibility

def plausibility_weight():
    """Training weight: the calibrated coefficient in the plausibility arms, 0 otherwise or before calibration."""
    return float(CAL[ARM]["coef"]) if (CAL is not None and ARM in JUDGE_ARMS and ARM in CAL) else 0.0

def _from_last(key, completions):
    v = _LAST_JUDGE.get(key) if _LAST_JUDGE.get("id") == id(completions) else None
    return [None] * len(completions) if v is None else [None if math.isnan(x) else float(x) for x in v]

def judge_mean(completions, **kwargs):
    return _from_last("mean", completions)

def judge_worst(completions, **kwargs):
    return _from_last("worst", completions)

def judge_sum(completions, **kwargs):
    return _from_last("sum", completions)

def judge_ratio(completions, **kwargs):
    return _from_last("ratio", completions)

def reasoning_tokens(completions, **kwargs):
    return [None if (r := tag(completion_text(c), "reasoning")) is None
            else float(len(tokenizer(r, add_special_tokens=False)["input_ids"])) for c in completions]

def solution_tokens(completions, **kwargs):
    return [None if (s := tag(completion_text(c), "solution")) is None
            else float(len(tokenizer(s, add_special_tokens=False)["input_ids"])) for c in completions]

# ---- the step monitor: prints this step's means and the judge's within-puzzle win rate, dumps completions every 20 steps
def within_puzzle_win_rate(keys, scores, correct):
    """Over (correct, wrong) pairs of the same puzzle (NaN scores left out): share where the correct one scores
    higher, ties 1/2. Returns (rate, pairs)."""
    wins = pairs = 0
    keys = np.array(keys)
    for k in set(keys.tolist()):
        s, c = scores[keys == k], correct[keys == k]
        cs, ws = s[c & ~np.isnan(s)], s[~c & ~np.isnan(s)]
        d = cs[:, None] - ws[None, :]
        wins += (d > 0).sum() + 0.5 * (d == 0).sum()
        pairs += d.size
    return (wins / pairs if pairs else float("nan")), int(pairs)

def make_step_monitor(owner):
    def step_monitor(prompts, completions, numbers, target, pid, **kwargs):
        correct = np.array(R.correctness_reward_func(prompts, completions, numbers, target)) > 0
        fmt = sum(np.array(f(completions=completions, numbers=numbers, target=target), dtype=float) for f in FORMAT_FUNCS)
        m = np.array(judge_mean(completions), dtype=float)
        cc = np.array([np.nan if x is None else x for x in R.caught_claims(completions)], dtype=float)
        st, gu = np.mean(R.stopped(completions)), np.mean(R.gave_up(completions))
        avg = lambda x: f"{np.nanmean(x):.2f}" if np.isfinite(x).any() else "nan"
        line = (f"sampled T {TEMPERATURE:g}  correct {correct.mean():.3f}  format {fmt.mean():.3f}"
                + (f"  stopped {st:.3f}" if st else "") + (f"  gave up {gu:.3f}" if gu else "")
                + f"  caught claims {avg(cc)}")
        rate, pairs = within_puzzle_win_rate(pid, m, correct)
        line += f"  | judge mean: correct {avg(m[correct])}  wrong {avg(m[~correct])}  win {rate:.2f} ({pairs})"
        if pairs and kwargs.get("log_metric"):
            kwargs["log_metric"]("judge_mean_win_rate", float(rate))
        if CAL is not None and "pass" in CAL:   # share above the pass threshold (no answer = fails), as the "pass" arm
            ok = (np.nan_to_num(m, nan=-np.inf) > CAL["pass"]["threshold"]).astype(float)
            line += f"  | pass: correct {avg(ok[correct])}  wrong {avg(ok[~correct])}"
        step = owner.state.global_step
        print(f"[step {step}] {line}")
        if step % 20 == 0:
            os.makedirs(LOG_DIR, exist_ok=True)
            with open(f"{LOG_DIR}/step_{step:05d}.txt", "w", encoding="utf-8") as f:
                f.write(f"=== Step {step} ===\n\n")
                for i, (p, c) in enumerate(zip(prompts, completions)):
                    f.write(f"### Sample {i + 1}\n[Scores] puzzle={pid[i]} correct={bool(correct[i])} "
                            f"judge_mean={m[i]:.4f}\n[Prompt]\n{p[-1]['content'].strip()}\n\n"
                            f"[Response]\n{completion_text(c).strip()}\n\n" + "=" * 80 + "\n\n")
        return [0.0] * len(completions)
    step_monitor.__name__ = "step_monitor"
    return step_monitor

# (function, weight); "plausibility" and "step_monitor" are bound to the trainer in [build]. Order matters: the judge
# monitors read the scores the plausibility reward just computed.
REWARD_FUNCS_TRAIN = ([(f, 0.0 if ARM == "intuitor" else 1.0) for f in FORMAT_FUNCS]   # Intuitor: its reward only
                      + ([("self_certainty", SC_WEIGHT)] if ARM == "intuitor" else [])
                      + [("plausibility", plausibility_weight()),
                         (R.correctness_reward_func, 1.0 if ARM == "correctness" else 0.0),
                         (judge_mean, 0.0), (judge_worst, 0.0), (judge_sum, 0.0), (judge_ratio, 0.0)]
                      + [(f, 0.0) for f in R.MONITORS + [reasoning_tokens, solution_tokens]]
                      + [("step_monitor", 0.0)])
print("reward functions (weight):", [(f if isinstance(f, str) else f.__name__, w) for f, w in REWARD_FUNCS_TRAIN])
print("calibration:", CALIBRATION_PATH if CAL else "none yet (run [reward calibration]; plausibility weight 0 until then)")

In [ ]:
# [vLLM sync fix]
# TRL syncs LoRA to vLLM by merging dW into the training model's bf16 W in place, streaming it, then unmerging.
# Each bf16 add/subtract rounds, so the "frozen" W drifts a little at every sync (seen: 0.16% of weights after 10 steps).
# This version never touches the training model: for each LoRA layer it builds bf16(W + dW) in a temporary tensor
# (dW and the sum in fp32, one rounding), streams it to vLLM, and frees it. Peak extra memory: one layer in fp32.
import torch
from accelerate.utils import is_peft_model
from peft.tuners.lora import LoraLayer
from trl.generation.vllm_generation import VLLMGeneration

if not hasattr(VLLMGeneration, "_trl_iter_named_params"):
    VLLMGeneration._trl_iter_named_params = VLLMGeneration._iter_named_params  # keep TRL's original

def _iter_named_params_out_of_place(self):
    model = self.model
    if not is_peft_model(model) or self.accelerator.num_processes > 1:
        yield from self._trl_iter_named_params()  # not the single-GPU LoRA case this fix is for
        return
    lora_layers = {name: m for name, m in model.named_modules() if isinstance(m, LoraLayer)}
    n_merged = 0
    for name, param in model.named_parameters():
        vllm_name = name.removeprefix("base_model.model.").replace(".base_layer", "")
        if model.prefix in vllm_name or "original_module" in vllm_name:
            continue  # LoRA A/B themselves don't exist in vLLM
        vllm_name = self._fix_param_name_to_vllm(vllm_name, extra_prefixes=["modules_to_save.default."])
        layer = lora_layers.get(name.rsplit(".base_layer.", 1)[0]) if name.endswith(".base_layer.weight") else None
        if layer is None:
            yield vllm_name, param.data
            continue
        assert not layer.merged, f"{name}: adapter is merged in the training model"
        with torch.no_grad():
            merged = param.data.float()
            for adapter in layer.active_adapters:
                if adapter in layer.lora_A:
                    merged += layer.get_delta_weight(adapter).float()
            merged = merged.to(param.dtype)
        n_merged += 1
        yield vllm_name, merged
        del merged
    if not getattr(self, "_out_of_place_reported", False):
        print(f"[vLLM sync fix] streamed {n_merged} LoRA layers as bf16(W + dW); training weights untouched")
        self._out_of_place_reported = True

VLLMGeneration._iter_named_params = _iter_named_params_out_of_place
print("VLLMGeneration._iter_named_params patched")

In [ ]:
# [eval] Greedy eval of any puzzle set on the trainer's vLLM engine (sync the weights first: before training the
# engine holds the base weights). Per completion: correct (check_answer), what is wrong (diagnose), format,
# truncation, the Countdown monitors. Used for the test set ([build] before, [train] after), the held-out strata and
# the held-out curve.
from vllm import SamplingParams

_FORMAT_RE = re.compile(r"<reasoning>.*</reasoning>\s*<solution>.*</solution>\s*<answer>.*</answer>", re.DOTALL)

def eval_puzzles(trainer, ds, tag_, out_dir=None, max_tokens=None, quiet=False):
    """Greedy completions for the puzzle Dataset ds; returns (summary dict, per-puzzle DataFrame)."""
    ids = [chat_prompt_ids(tokenizer, p) for p in ds["prompt"]]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(temperature=0.0, max_tokens=max_tokens or MAX_COMPLETION_LENGTH), use_tqdm=not quiet)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    texts = [o.outputs[0].text for o in outs]
    wrapped = [[{"role": "assistant", "content": t}] for t in texts]
    df = pd.DataFrame({"pid": ds["pid"], "numbers": [json.dumps(n) for n in ds["numbers"]], "target": ds["target"],
                       "completion": texts, "tokens": [len(o.outputs[0].token_ids) for o in outs],
                       "truncated": [o.outputs[0].finish_reason == "length" for o in outs],
                       "format_ok": [bool(_FORMAT_RE.search(t)) for t in texts]})
    df["answer"] = [tag(t, "answer") for t in texts]
    df["correct"] = np.array(R.correctness_reward_func(None, wrapped, ds["numbers"], ds["target"])) > 0
    df["kind"] = [diagnose(a, n, t, tr) for a, n, t, tr in zip(df.answer, ds["numbers"], ds["target"], df.truncated)]
    for f in R.MONITORS:
        df[f.__name__] = np.array(f(completions=wrapped, numbers=ds["numbers"], target=ds["target"]), dtype=float)
    summary = {"tag": tag_, "n": len(df), "accuracy": df.correct.mean(),
               "accuracy_finished": df.loc[~df.truncated, "correct"].mean(), "truncated": df.truncated.mean(),
               "format_ok": df.format_ok.mean(), "tokens": df.tokens.mean(),
               **{f.__name__: df[f.__name__].mean() for f in R.MONITORS},
               **{f"kind/{k}": v for k, v in df.kind.value_counts(normalize=True).items()}}
    if not quiet:
        print(f"[eval:{tag_}] greedy, n={len(df)}  accuracy {summary['accuracy']:.3f} (finished-only "
              f"{summary['accuracy_finished']:.3f})  truncated {summary['truncated']:.3f}  format {summary['format_ok']:.3f}  "
              f"tokens {summary['tokens']:.0f}\n  " + ", ".join(f"{k[5:]} {v:.2f}" for k, v in summary.items()
                                                                 if k.startswith("kind/") and k != "kind/correct"))
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
        json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer, bool)) else v) for k, v in summary.items()},
                  open(f"{out_dir}/eval_{tag_}.json", "w"), indent=2)
        df.to_csv(f"{out_dir}/eval_{tag_}_samples.csv", index=False)
    return summary, df

def pass_at_k(trainer, ds, tag_, k, out_dir=None, strata=None, temperature=None):
    """k sampled completions per puzzle at temperature (default PASS_TEMPERATURE): pass@1 (mean accuracy), pass@k (any correct),
    distinct correct answers per puzzle; overall and per stratum (strata: pid -> name). Saves eval_{tag_}_pass{k}.*"""
    ids = [chat_prompt_ids(tokenizer, p) for p in ds["prompt"]]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(n=k, temperature=temperature or PASS_TEMPERATURE, max_tokens=MAX_COMPLETION_LENGTH, seed=0), use_tqdm=True)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    rows = []
    for o, pid, nums, tgt in zip(outs, ds["pid"], ds["numbers"], ds["target"]):
        texts = [c.text for c in o.outputs]
        ok = np.array(R.correctness_reward_func(None, [[{"role": "assistant", "content": t}] for t in texts],
                                                [nums] * k, [tgt] * k)) > 0
        right = {re.sub(r"\s+", "", tag(t, "answer") or "") for t, c in zip(texts, ok) if c}
        rows.append({"pid": pid, "stratum": (strata or {}).get(pid, "other"), "correct": int(ok.sum()),
                     "pass1": ok.mean(), f"pass{k}": float(ok.any()), "distinct_correct": len(right),
                     "truncated": float(np.mean([c.finish_reason == "length" for c in o.outputs]))})
    df = pd.DataFrame(rows)
    cols = ["pass1", f"pass{k}", "distinct_correct", "truncated"]
    summary = pd.concat([df[cols].mean().rename("total").to_frame().T, df.groupby("stratum")[cols].mean()])
    summary["n"] = [len(df)] + df.groupby("stratum").size().tolist()
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
        df.to_csv(f"{out_dir}/eval_{tag_}_pass{k}_samples.csv", index=False)
        summary.to_csv(f"{out_dir}/eval_{tag_}_pass{k}.csv")
    return summary, df

def eval_mixed(trainer, tag_, out_dir=None, test4=None):
    """Greedy accuracy on Gandhi et al. 2025's mix: test3_set (3-number) and the 4-number test set, mixed = their mean.
    test4: an existing (summary, df) greedy eval of the SAME weights on the full test_set; reused (saved again as
    test4_{tag_}) instead of a second eval of the same puzzles. Returns (mixed summary, 3-number summary, 4-number summary)."""
    s3, _ = eval_puzzles(trainer, test3_set, f"test3_{tag_}", out_dir=out_dir, quiet=True)
    if test4 is not None and len(test4[1]) == len(test_set):
        s4, d4 = {**test4[0], "tag": f"test4_{tag_}"}, test4[1]
        if out_dir is not None:
            json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer, bool)) else v) for k, v in s4.items()},
                      open(f"{out_dir}/eval_test4_{tag_}.json", "w"), indent=2)
            d4.to_csv(f"{out_dir}/eval_test4_{tag_}_samples.csv", index=False)
    else:
        s4, _ = eval_puzzles(trainer, test_set, f"test4_{tag_}", out_dir=out_dir, quiet=True)
    mixed = {"3-number": s3["accuracy"], "4-number": s4["accuracy"], "mixed": (s3["accuracy"] + s4["accuracy"]) / 2,
             "truncated 3": s3["truncated"], "truncated 4": s4["truncated"]}
    return mixed, s3, s4

# ---- reporting: one table per eval point; the weights it measures are named in the header ----
CURRENT_WEIGHTS = globals().get("CURRENT_WEIGHTS", "starting adapter")   # what the session's policy holds ([train] sets it)

def curve_of(out_dir, tag_):
    """Greedy summary of the held-out curve puzzles (CURVE_STRATA, [held-out]) from an eval_heldout(..., tag_, out_dir)."""
    df = pd.concat([pd.read_csv(f"{out_dir}/eval_heldout_{s}_{tag_}_samples.csv") for s in CURVE_STRATA])
    fin = ~df.truncated.astype(bool)
    return {"accuracy": float(df.correct.mean()), "accuracy_finished": float(df.loc[fin, "correct"].mean()),
            "truncated": float(df.truncated.mean()), "format_ok": float(df.format_ok.mean()),
            "tokens": float(df.tokens.mean()), "n": len(df)}

def _mixed_before_dir():
    return f"{GRPO_DIR}/eval_mixed_before_n{len(test3_set)}_max{MAX_COMPLETION_LENGTH}"

def mixed_before_cached():
    """The starting adapter's 3- and 4-number test evals (greedy) if cached, else None."""
    d = _mixed_before_dir()
    if not os.path.exists(f"{d}/summary.json"):
        return None
    return {k: json.load(open(f"{d}/eval_{k}_before.json")) for k in ("test3", "test4")}

def mixed_before_evals(trainer):
    """As mixed_before_cached; if not cached, evals the starting adapter (loaded for the eval, then the session's
    weights are restored) and caches it in GRPO_DIR."""
    if mixed_before_cached() is None:
        from peft import get_peft_model_state_dict, set_peft_model_state_dict
        from safetensors.torch import load_file
        d = _mixed_before_dir()
        _now = {k: v.detach().clone() for k, v in get_peft_model_state_dict(trainer.model, adapter_name="default").items()}
        set_peft_model_state_dict(trainer.model, load_file(f"{_policy_init}/adapter_model.safetensors"), adapter_name="default")
        trainer.vllm_generation.sync_weights()
        try:
            mixed, _, _ = eval_mixed(trainer, "before", d)
        finally:
            set_peft_model_state_dict(trainer.model, _now, adapter_name="default")
            trainer.vllm_generation.sync_weights()
        json.dump(mixed, open(f"{d}/summary.json", "w"), indent=2)
    return mixed_before_cached()

_SETS = (("test4", "4-number test"), ("test3", "3-number test"), ("curve", "held-out curve puzzles"))

def report(title, weights, cols, out=None):
    """Prints ONE table: rows = (set, decoding, puzzles x samples, metric), columns = cols' keys (e.g. "start", "step 50").
    cols: {column: results}; results: dict with any of "test4", "test3", "curve" (greedy eval summaries) and "pass"
    (pass_at_k summary, sampled). Saved to out (csv) if given."""
    rows = {}
    for key, name in _SETS:
        for c, res in cols.items():
            s = (res or {}).get(key)
            if s is None:
                continue
            for m in ("accuracy", "truncated"):
                rows.setdefault((name, "greedy", f"{int(s.get('n', 0))} x 1", m), {})[c] = float(s[m])
    for c, res in cols.items():
        p = (res or {}).get("pass")
        if p is None:
            continue
        t = p.loc["total"]
        for m, lab in (("pass1", "pass@1"), (f"pass{PASS_AT_K}", f"pass@{PASS_AT_K}"),
                       ("distinct_correct", "distinct correct answers"), ("truncated", "truncated")):
            rows.setdefault((f"pass@{PASS_AT_K} puzzles (in curve set)", f"T {PASS_TEMPERATURE:g}",
                             f"{int(t['n'])} x {PASS_AT_K}", lab), {})[c] = float(t[m])
    if not rows:
        print(f"\n[report] {title} | weights: {weights}: nothing to report")
        return None
    df = pd.DataFrame.from_dict(rows, orient="index").reindex(columns=list(cols))
    df.index = pd.MultiIndex.from_tuples(df.index, names=["set", "decoding", "puzzles x samples", "metric"])
    print(f"\n[report] {title} | weights: {weights}")
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
        print(df.to_string(na_rep="-"))
    if out is not None:
        df.to_csv(out)
    return df

def curve_line(curve):
    """The held-out curve as one line: step -> greedy accuracy."""
    n = len(CURVE_SET) if "CURVE_SET" in globals() else "?"
    return (f"[held-out curve] held-out curve puzzles, greedy, {n} x 1, accuracy by step: "
            + "  ".join(f"{int(r['step'])}: {r['accuracy']:.3f}" for r in curve))

In [ ]:
# [build] GRPO config + trainer (rewards from [rewards]) + the "before" test eval (cached in GRPO_DIR). Then: [pool] ->
# [held-out] -> [reward calibration] -> [early stop] -> [train]. Settings as Consistency_GRPO's 8B run b.
import csv, shutil, tempfile
from datetime import datetime
from transformers import TrainerCallback
from trl import GRPOTrainer, GRPOConfig
import passk                     # TRAIN_PASS_K: the analytical pass@k advantages (code/countdown/passk.py)
importlib.reload(passk)

class CSVLoggerCallback(TrainerCallback):
    """Every log row to a CSV; the header grows when new keys appear (rewritten in place)."""
    def __init__(self, path):
        self.path, self.fields = path, None
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if os.path.exists(path) and os.path.getsize(path):
            with open(path, newline="") as f:
                self.fields = next(csv.reader(f))
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        logs = {"step": state.global_step, **logs}
        if self.fields is None:
            self.fields = list(logs)
            with open(self.path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()
        new = [k for k in logs if k not in self.fields]
        if new:
            self.fields += sorted(new)
            rows = list(csv.DictReader(open(self.path, newline="")))
            fd, tmp = tempfile.mkstemp(dir=os.path.dirname(self.path))
            os.close(fd)
            with open(tmp, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=self.fields)
                w.writeheader()
                w.writerows(rows)
            shutil.move(tmp, self.path)
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(logs)

# CONTINUE_FROM: seed LOG_DIR with the source checkpoint and its rows up to that step (once: not if LOG_DIR has checkpoints)
if CONTINUE_FROM:
    _src_dir, _src_step = f"{GRPO_DIR}/runs/{CONTINUE_FROM[0]}", int(CONTINUE_FROM[1])
    assert os.path.isdir(f"{_src_dir}/checkpoint-{_src_step}"), f"no checkpoint-{_src_step} in {_src_dir}"
    assert f"{_src_dir}" != LOG_DIR, "CONTINUE_FROM needs a new RUN_TAG"
    assert MAX_STEPS > _src_step, (MAX_STEPS, _src_step)
    if not any(n.startswith("checkpoint-") for n in (os.listdir(LOG_DIR) if os.path.isdir(LOG_DIR) else [])):
        os.makedirs(LOG_DIR, exist_ok=True)
        shutil.copytree(f"{_src_dir}/checkpoint-{_src_step}", f"{LOG_DIR}/checkpoint-{_src_step}")
        for name in ("heldout_curve.csv", "pass_curve.csv", "metrics.csv"):
            if os.path.exists(f"{_src_dir}/{name}"):
                _d = pd.read_csv(f"{_src_dir}/{name}")
                _d[_d.step <= _src_step].to_csv(f"{LOG_DIR}/{name}", index=False)
        json.dump({"continue_from": CONTINUE_FROM, "lr_schedule": LR_SCHEDULE},
                  open(f"{LOG_DIR}/continued_from.json", "w"), indent=1)
        print(f"[continue] copied checkpoint-{_src_step} and rows <= {_src_step} from {_src_dir}")
    else:
        print(f"[continue] {LOG_DIR} already has checkpoints: resuming from the newest")

training_args = GRPOConfig(
    use_vllm=True, vllm_mode="colocate", vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    vllm_max_model_length=MAX_COMPLETION_LENGTH + 512,     # prompt (~110 tokens) + completion
    gradient_checkpointing=True,
    loss_type=LOSS_TYPE, scale_rewards=SCALE_REWARDS, max_grad_norm=MAX_GRAD_NORM, epsilon=EPSILON, temperature=TEMPERATURE,
    beta=KL_BETA,
    reward_weights=[w for _, w in REWARD_FUNCS_TRAIN],
    learning_rate=LEARNING_RATE, adam_beta1=0.9, adam_beta2=ADAM_BETA2, weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS, lr_scheduler_type=LR_SCHEDULE, optim="adamw_torch",
    logging_strategy="steps", logging_steps=5,
    save_strategy="steps", save_steps=25, save_total_limit=None,   # every checkpoint kept: [early stop] picks the best
    output_dir=LOG_DIR, report_to=["tensorboard"],
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=COMPLETIONS_PER_STEP // MICRO_BATCH,
    num_generations=NUM_GENERATIONS, max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=10 if SMOKE_RUN else MAX_STEPS, eval_strategy="no",
)

def block_keep_mask(completion_ids, completion_mask, tok, blocks):
    """1 for every completion token except those inside the last <block> ... </block> of each of `blocks` (the tags and
    everything outside stay 1); same shape and dtype as completion_mask."""
    keep = torch.ones_like(completion_mask)
    for i in range(completion_ids.shape[0]):
        toks = completion_ids[i][: int(completion_mask[i].sum())].tolist()
        pieces = [tok.decode([t]) for t in toks]
        if "".join(pieces) != tok.decode(toks):      # multi-byte characters split across tokens: offsets by prefix
            starts = [len(tok.decode(toks[:k])) for k in range(len(toks) + 1)]
            text = tok.decode(toks)
        else:
            starts = [0]
            for p in pieces:
                starts.append(starts[-1] + len(p))
            text = "".join(pieces)
        for name in blocks:
            s = text.rfind(f"<{name}>")
            e = text.find(f"</{name}>", s) if s >= 0 else -1
            if s < 0 or e < 0:
                continue
            cs = s + len(f"<{name}>")
            for k in range(len(toks)):
                if starts[k] >= cs and starts[k + 1] <= e:   # tokens fully inside the block's text
                    keep[i, k] = 0
    return keep

class CountdownGRPOTrainer(GRPOTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # the judge and the step monitor need the trainer (model, tokenizer, step): bind them now
        for i, f in enumerate(self.reward_funcs):
            if getattr(f, "__name__", "") == "plausibility":
                self.reward_funcs[i] = make_plausibility(self)
            elif getattr(f, "__name__", "") == "step_monitor":
                self.reward_funcs[i] = make_step_monitor(self)
            elif getattr(f, "__name__", "") == "self_certainty":
                self.reward_funcs[i] = intuitor.make_self_certainty(self, chat_prompt_ids)
            elif getattr(f, "__name__", "") == "correctness_reward_func" and TRAIN_PASS_K:
                self.reward_funcs[i] = self._recording(f)
        assert not TRAIN_PASS_K or any(getattr(f, "__qualname__", "").startswith("CountdownGRPOTrainer._recording")
            for f in self.reward_funcs), "TRAIN_PASS_K: no correctness_reward_func to record"

    def _recording(self, f):
        """f, keeping its last output (this generation batch's correctness, for TRAIN_PASS_K); same name, same logs."""
        def correctness_reward_func(*args, **kwargs):
            self._last_correctness = f(*args, **kwargs)
            return self._last_correctness
        return correctness_reward_func

    def _generate_and_score_completions(self, inputs):
        out = super()._generate_and_score_completions(inputs)
        if TRAIN_PASS_K:
            # with SCALE_REWARDS False, TRL's advantage is sum_f w_f (r_f - group mean of r_f): take out the correctness
            # term and put in the pass@k advantage of each group (groups are contiguous, one GPU: the whole batch)
            a = out["advantages"]
            names = [getattr(f, "__name__", "") for f in self.reward_funcs]
            w = float(self.reward_weights[names.index("correctness_reward_func")])
            r = torch.tensor(self._last_correctness, dtype=a.dtype, device=a.device)
            assert r.shape == a.shape and self.accelerator.num_processes == 1, "pass@k: expected the whole batch on one GPU"
            g = r.view(-1, self.num_generations)
            pk = [passk.passk_advantages((row > 0).tolist(), TRAIN_PASS_K) for row in g]
            out["advantages"] = a - (w * (g - g.mean(dim=1, keepdim=True))).view(-1) + torch.tensor(pk, dtype=a.dtype, device=a.device).view(-1)
            m = self._metrics["train" if self.model.training else "eval"]
            m["passk/group_reward"].append(float(np.mean([passk.passk_group_reward((row > 0).tolist(), TRAIN_PASS_K) for row in g])))
            m["passk/zero_adv_groups"].append(float(np.mean([not any(x) for x in pk])))
        if POSITIVE_ADV_SCALE != 1.0:   # negative-sample reinforcement: shrink only the positive advantages
            a = out["advantages"]
            out["advantages"] = torch.where(a > 0, POSITIVE_ADV_SCALE * a, a)
        if MASK_GRAD_BLOCKS:
            # TRL's loss mask is completion_mask * tool_mask (the hook it uses for tool outputs); the DAPO loss is
            # divided by num_items_in_batch, computed before this, so recompute it from the new mask
            keep = block_keep_mask(out["completion_ids"], out["completion_mask"], self.processing_class, MASK_GRAD_BLOCKS)
            out["tool_mask"] = keep if out.get("tool_mask") is None else out["tool_mask"] * keep
            loss_mask = out["completion_mask"] * out["tool_mask"]
            out["num_items_in_batch"] = self.accelerator.gather(loss_mask.sum()).sum()
            masked = (out["completion_mask"] * (1 - keep)).sum() / out["completion_mask"].sum().clamp(min=1)
            self._metrics["train" if self.model.training else "eval"]["grad_masked_frac"].append(float(masked))
        return out

    def log(self, logs, start_time=None):
        super().log(logs, start_time)
        if "reward" in logs and "train_runtime" not in logs:
            v = lambda k, d=float("nan"): d if logs.get(k) is None else float(logs[k])   # TRL logs None for all-NaN
            print(f"[{self.state.global_step}] loss={v('loss', 0.0):.4f}, reward={v('reward'):.4f}, "
                  f"grad_norm={v('grad_norm'):.3f}, entropy={v('entropy'):.3f}, "
                  f"correct={v('rewards/correctness_reward_func/mean') / 2:.3f}, "
                  f"reasoning={v('rewards/reasoning_tokens/mean'):.0f}")

def _placeholder(name):
    f = lambda completions, **kwargs: [0.0] * len(completions)
    f.__name__ = name
    return f

trainer = CountdownGRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args, train_dataset=candidates,   # [pool] narrows it
    reward_funcs=[_placeholder(f) if isinstance(f, str) else f for f, _ in REWARD_FUNCS_TRAIN])
trainer.add_callback(CSVLoggerCallback(f"{LOG_DIR}/metrics.csv"))
print("reward weights:", {getattr(f, "__name__", "?"): float(w) for f, w in zip(trainer.reward_funcs, trainer.reward_weights)})

# The test set "before" (the starting adapter, greedy) doesn't depend on the arm: cached in GRPO_DIR, copied per run.
EVAL_MAX = 200 if not SMOKE_RUN else 50
_test = test_set.select(range(min(EVAL_MAX, len(test_set))))

# CONTINUE_FROM: the policy starts at the resume checkpoint from here on (the sampler check, [train]), not only once the
# trainer resumes: in cont100 (2026-10-22) the trainer's resume restored step, optimizer and scheduler but trained the
# SFT adapter. Loaded after the trainer is built, so TRL's KL reference ("ref", copied from the policy at build) stays
# the starting adapter (checked). The "before" evals stay the starting adapter's: they must come from their caches.
if CONTINUE_FROM:
    from peft import get_peft_model_state_dict, set_peft_model_state_dict
    from safetensors.torch import load_file
    _resume_ckpt = max((n for n in os.listdir(LOG_DIR) if n.startswith("checkpoint-")), key=lambda n: int(n.rsplit("-", 1)[1]))
    _want = load_file(f"{LOG_DIR}/{_resume_ckpt}/adapter_model.safetensors")
    _res = set_peft_model_state_dict(trainer.model, _want, adapter_name="default")
    assert not _res.unexpected_keys, _res.unexpected_keys[:4]
    _same = lambda now, want: set(now) == set(want) and all(torch.equal(now[k].cpu(), want[k].to(now[k].dtype)) for k in want)
    assert _same(get_peft_model_state_dict(trainer.model, adapter_name="default"), _want), f"policy != {_resume_ckpt}"
    if KL_BETA:
        assert _same(get_peft_model_state_dict(trainer.model, adapter_name="ref"),
                     load_file(f"{_policy_init}/adapter_model.safetensors")), "KL reference is not the starting adapter"
    assert all(p.requires_grad for n, p in trainer.model.named_parameters() if ".default." in n and "lora_" in n)
    assert os.path.exists(f"{GRPO_DIR}/eval_test_before_n{len(_test)}_max{MAX_COMPLETION_LENGTH}.json"), \
        "no cached test 'before' eval: it would evaluate the checkpoint as 'before'"
    print(f"[continue] policy = {os.path.basename(LOG_DIR)}/{_resume_ckpt} (KL reference: the starting adapter)")
trainer.vllm_generation.sync_weights()      # the engine starts with base weights

# Sampler check: vLLM (what generates) and the trainer (what computes the policy's log-probs) must be the same model.
# Greedy completions of a few test puzzles, each token's log-prob from both. Runs a-c: mean |difference| ~0.003 per
# token (bf16); the merged-judge version of 2026-10-16 had ~0.012 and an importance-sampling ratio of ~0.4.
SAMPLER_MAX_DIFF = 0.006
def sampler_check(n_prompts=8, max_tokens=128):
    ds = _test.select(range(min(n_prompts, len(_test))))
    ids = [chat_prompt_ids(tokenizer, p) for p in ds["prompt"]]
    outs = trainer.vllm_generation.llm.generate([{"prompt_token_ids": x} for x in ids],
                                                SamplingParams(temperature=0.0, max_tokens=max_tokens, logprobs=1),
                                                use_tqdm=False)
    diffs = []
    for x, o in zip(ids, outs):
        toks = list(o.outputs[0].token_ids)
        v = np.array([lp[t].logprob for lp, t in zip(o.outputs[0].logprobs, toks)])
        inp = torch.tensor([x + toks], device=trainer.model.device)
        with torch.no_grad():
            logits = trainer.model(input_ids=inp).logits[0, len(x) - 1:-1].float()
        hf = torch.log_softmax(logits, -1).gather(1, inp[0, len(x):, None])[:, 0].cpu().numpy()
        diffs.append(np.abs(hf - v))
    d = np.concatenate(diffs)
    return float(d.mean()), float(d.max()), len(d)
_sc_mean, _sc_max, _sc_n = sampler_check()
print(f"[sampler check] vLLM vs trainer log-probs over {_sc_n} tokens: mean |diff| {_sc_mean:.4f}, max {_sc_max:.3f} "
      f"(runs a-c ~0.003)")
assert _sc_mean < SAMPLER_MAX_DIFF, (f"vLLM and the trainer disagree ({_sc_mean:.4f} > {SAMPLER_MAX_DIFF}): the weights "
                                     f"vLLM generates with are not the policy the trainer updates")
_before = f"{GRPO_DIR}/eval_test_before_n{len(_test)}_max{MAX_COMPLETION_LENGTH}"
if os.path.exists(f"{_before}.json"):
    eval_before = json.load(open(f"{_before}.json"))
    print(f"[eval:before] 4-number test, greedy, {int(eval_before['n'])} x 1, starting adapter (cached {_before}.json): "
          f"accuracy {eval_before['accuracy']:.3f}, truncated {eval_before['truncated']:.3f}")
else:
    eval_before, _ = eval_puzzles(trainer, _test, "test_before", out_dir=GRPO_DIR)
    for ext in (".json", "_samples.csv"):
        shutil.move(f"{GRPO_DIR}/eval_test_before{ext}", f"{_before}{ext}")
os.makedirs(LOG_DIR, exist_ok=True)
for ext in (".json", "_samples.csv"):
    shutil.copy(f"{_before}{ext}", f"{LOG_DIR}/eval_test_before{ext}")

In [ ]:
# [pool] (USE_POOL only) Pass rate of the starting adapter on each candidate (PASS_SAMPLES samples at the training
# temperature and length cap); training uses the candidates with a pass rate in PASS_BAND. Cached in POOL_FILE, so all
# arms train on the same puzzles. With USE_POOL = False nothing is sampled: training uses all candidates.
lo, hi = PASS_BAND
if not USE_POOL:
    pool = band = None
    print(f"[pool] skipped (USE_POOL = False): training on all {len(candidates)} candidates minus [held-out]'s random set")
elif os.path.exists(POOL_FILE):
    pool = pd.read_csv(POOL_FILE)
    print(f"[pool] cached: {POOL_FILE}")
else:
    trainer.vllm_generation.sync_weights()      # generate with the starting policy, not vLLM's base weights
    ids = [chat_prompt_ids(tokenizer, p) for p in candidates["prompt"]]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(n=PASS_SAMPLES, temperature=trainer.args.temperature, max_tokens=MAX_COMPLETION_LENGTH,
                       seed=POOL_SEED), use_tqdm=True)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    pool = pd.DataFrame({
        "pid": candidates["pid"],
        "pass_rate": [np.mean([bool(a) and check_answer(a, n, t) for a in (tag(c.text, "answer") for c in o.outputs)])
                      for o, n, t in zip(outs, candidates["numbers"], candidates["target"])],
        "truncated": [np.mean([c.finish_reason == "length" for c in o.outputs]) for o in outs],
        "tokens": [np.mean([len(c.token_ids) for c in o.outputs]) for o in outs]})
    os.makedirs(GRPO_DIR, exist_ok=True)
    pool.to_csv(POOL_FILE, index=False)
    print(f"[pool] saved {POOL_FILE}")

if USE_POOL:
    band = pool[(pool.pass_rate >= lo) & (pool.pass_rate <= hi)]
    print(f"pass rate over {len(pool)} candidates: mean {pool.pass_rate.mean():.3f}; "
          f"distribution {pool.pass_rate.round(3).value_counts().sort_index().to_dict()}")
    print(f"truncated {pool.truncated.mean():.3f}; mean completion tokens {pool.tokens.mean():.0f}")
    _prompts_per_step = trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps // trainer.args.num_generations
    print(f"band {PASS_BAND}: {len(band)} puzzles; training sees {trainer.args.max_steps} steps x {_prompts_per_step} "
          f"prompts = {trainer.args.max_steps * _prompts_per_step} prompts")

In [ ]:
# [held-out] Eval frames that are never trained on (fixed, seed HELDOUT_SEED; saved in SPLIT_OUT, shared by all arms).
# USE_POOL = True:
#   band_heldout  puzzles from the pass-rate band, removed from training here (same difficulty as training)
#   hard          candidates with pass rate 0 (does training transfer to puzzles it couldn't solve?)
#   easy          candidates with pass rate 1
#   unscored      the UNSCORED_N puzzles never sampled by [pool]
# USE_POOL = False:
#   heldout       a random set of the candidates, removed from training here (same distribution as training)
#   unscored      the UNSCORED_N extra puzzles of the RL draw
# The test set (Consistency_Countdown's 200) stays the main before/after eval. Greedy, MAX_COMPLETION_LENGTH.
from datasets import concatenate_datasets
HELDOUT_BAND, HELDOUT_EASY, HELDOUT_HARD, HELDOUT_SEED = 100, 100, 100, 1
SPLIT_OUT = (f"{GRPO_DIR}/grpo_split_n{POOL_CANDIDATES}_s{PASS_SAMPLES}_seed{POOL_SEED}_band{lo:g}-{hi:g}.json" if USE_POOL
             else f"{GRPO_DIR}/grpo_split_n{POOL_CANDIDATES}_nopool_seed{HELDOUT_SEED}.json")

if os.path.exists(SPLIT_OUT):
    grpo_split = json.load(open(SPLIT_OUT))
    print(f"[held-out] split loaded: {SPLIT_OUT}")
else:
    r = np.random.default_rng(HELDOUT_SEED)
    pick = lambda ids, n: sorted(r.choice(sorted(ids), size=min(n, len(ids)), replace=False).tolist())
    if USE_POOL:
        band_ids = set(band.pid)
        grpo_split = {"band_heldout": pick(band_ids, min(HELDOUT_BAND, len(band_ids) // 5)),   # at most 1/5 of the band
                      "easy": pick(set(pool.loc[pool.pass_rate == 1, "pid"]), HELDOUT_EASY),
                      "hard": pick(set(pool.loc[pool.pass_rate == 0, "pid"]), HELDOUT_HARD),
                      "unscored": sorted(unscored["pid"])}
        grpo_split["train"] = sorted(band_ids - set(grpo_split["band_heldout"]))
        grpo_split["note"] = (f"puzzle ids (pid) of the RL draw; training = train (pass rate in {PASS_BAND} under "
                              f"{ADAPTER_PATH}" + (f" / start {POLICY_START} on the fused DPO judge" if JUDGE == "dpo" else "")
                              + "); the other lists are held out. The test set is separate.")
    else:
        all_ids = set(candidates["pid"])
        grpo_split = {"heldout": pick(all_ids, min(HELDOUT_BAND, len(all_ids) // 5)), "unscored": sorted(unscored["pid"])}
        grpo_split["train"] = sorted(all_ids - set(grpo_split["heldout"]))
        grpo_split["note"] = ("puzzle ids (pid) of the RL draw; training = train (all candidates, no pass-rate filter); "
                              "heldout and unscored are held out. The test set is separate.")
    json.dump(grpo_split, open(SPLIT_OUT, "w"))
    print(f"[held-out] split saved: {SPLIT_OUT}")

_all = {**{p: i for i, p in enumerate(candidates["pid"])}}
train_dataset = candidates.select([_all[p] for p in grpo_split["train"]])
if train3 is not None:                            # TRAIN_MIX_3: the 3-number training puzzles ([common])
    train_dataset = concatenate_datasets([train_dataset, train3])
trainer.train_dataset = train_dataset           # the dataloader is built in trainer.train(), from this dataset
HELDOUT_STRATA = ["band_heldout", "easy", "hard", "unscored"] if USE_POOL else ["heldout", "unscored"]
CURVE_STRATA = ["band_heldout" if USE_POOL else "heldout", "unscored"]   # the held-out curve of [early stop]

def heldout_dataset(stratum):
    if stratum == "unscored":
        return unscored
    return candidates.select([_all[p] for p in grpo_split[stratum]])

def eval_heldout(trainer, tag_, out_dir):
    """Greedy eval per held-out stratum; one row per stratum."""
    rows = {}
    for s in HELDOUT_STRATA:
        if len(heldout_dataset(s)) == 0:      # e.g. no pass-rate-1 puzzles in a small (smoke) pool
            continue
        summ, df = eval_puzzles(trainer, heldout_dataset(s), f"heldout_{s}_{tag_}", out_dir=out_dir, quiet=True)
        rows[s] = {k: summ[k] for k in ("accuracy", "accuracy_finished", "truncated", "format_ok", "tokens", "n")}
    return pd.DataFrame(rows).T

print({k: len(grpo_split[k]) for k in HELDOUT_STRATA + ["train"]}, "-> training on", len(train_dataset),
      f"({len(train3)} of them 3-number)" if train3 is not None else "")
_before_dir = f"{GRPO_DIR}/heldout_before_{os.path.basename(SPLIT_OUT)[:-5]}"
if os.path.exists(f"{_before_dir}/summary.csv"):
    heldout_before = pd.read_csv(f"{_before_dir}/summary.csv", index_col=0)
    print(f"[held-out] 'before' cached: {_before_dir}")
else:
    assert not CONTINUE_FROM, f"CONTINUE_FROM: no cached held-out 'before' in {_before_dir} (it would eval the checkpoint)"
    trainer.vllm_generation.sync_weights()      # the starting adapter, not vLLM's base weights
    heldout_before = eval_heldout(trainer, "before", _before_dir)
    heldout_before.to_csv(f"{_before_dir}/summary.csv")
# (per-stratum numbers: heldout_before, saved in _before_dir/summary.csv)

# sampled pass@k on the curve puzzles (CURVE_STRATA), starting adapter: cached next to the greedy "before" eval
from datasets import concatenate_datasets
_per = PASS_N // len(CURVE_STRATA)                   # the first _per puzzles of each curve stratum (a fixed set)
PASS_SET = concatenate_datasets([heldout_dataset(s).select(range(min(_per, len(heldout_dataset(s))))) for s in CURVE_STRATA])
PASS_STRATA = {p: s for s in CURVE_STRATA for p in heldout_dataset(s).select(range(min(_per, len(heldout_dataset(s)))))["pid"]}
if PASS_AT_K:
    _pk = f"{_before_dir}/eval_before_n{len(PASS_SET)}_pass{PASS_AT_K}.csv"
    if os.path.exists(_pk):
        pass_before = pd.read_csv(_pk, index_col=0)
    else:
        assert not CONTINUE_FROM, f"CONTINUE_FROM: no cached pass@k 'before' in {_before_dir} (it would sample the checkpoint)"
        trainer.vllm_generation.sync_weights()
        pass_before, _ = pass_at_k(trainer, PASS_SET, f"before_n{len(PASS_SET)}", PASS_AT_K, out_dir=_before_dir,
                                   strata=PASS_STRATA)
else:
    pass_before = None
_mb0 = mixed_before_cached()
report("starting adapter", "starting adapter (cached evals)",
       {"start": {"test4": eval_before, "test3": _mb0["test3"] if _mb0 else None,
                  "curve": curve_of(_before_dir, "before"), "pass": pass_before}})

In [ ]:
# [reward calibration] Run after [held-out], before [train]. Inference only. Samples real GRPO groups (CAL_PROMPTS
# training puzzles x num_generations, the starting adapter), scores them with the judge (raw mean, sum and worst) and
# the format rewards, and sets for each judge arm (JUDGE_ARMS; saved to CALIBRATION_PATH, shared by all arms):
#   floor    1st percentile of real scores (lower scores are clipped to it: caps outliers)
#   missing  two levels, chosen at reward time by MISSING_AT: "below_floor" (5% of the median-to-floor distance below
#            the floor: no solution / over the cap is worse than any scored answer) and "wrong_median" (the median value
#            of the scored wrong answers)
#   pass     (the "pass" arm) threshold on the raw mean: PASS_THRESHOLD, or the (1 - PASS_MAX_FALSE_PASS) quantile of
#            the scored wrong answers; coef from the spread of the 0/1 value over all completions (missing = 0)
#   coef     TARGET_RATIO x the anchor's within-group spread / the value's within-group spread (with
#            scale_rewards=False a reward's push inside a group is proportional to that spread). Anchor (SPREAD_ANCHOR):
#            "correctness" (default): the correctness reward's spread on the same groups, so the plausibility arms push
#            as hard as the correctness arm (never 0: the pool keeps puzzles with right and wrong answers);
#            "format": the format rewards' spread, as on GSM8K (small here, and 0 once the format saturates)
#   mean_worst also: mean / SD of the clipped mean and worst (the z-scores), and the z for a solution without "=" lines
#   ratio    (JUDGE "dpo") also a ceiling, the 99th percentile (sum - ref_sum has no natural top)
# A file without a ratio entry is redone in full; the old one is kept as <name>_before_ratio.json first.
# Also reports the judge's within-puzzle win rate on these groups (as a check against Consistency_Countdown).
# Skipped when CALIBRATION_PATH has every judge arm (RECALIBRATE = True redoes it; the sampling is seeded).
RECALIBRATE = False
_saved_cal = json.load(open(CALIBRATION_PATH)) if os.path.exists(CALIBRATION_PATH) else None
_cal_complete = lambda c: (all(k in c for k in JUDGE_ARMS)
                           and all("missing_wrong_median" in c[k] for k in ("mean", "sum", "mean_worst") + (("ratio",) if "ratio" in JUDGE_ARMS else ()))
                           and c["pass"].get("threshold_setting") == [PASS_THRESHOLD, PASS_MAX_FALSE_PASS])   # JSON: a list
if _saved_cal is not None and not RECALIBRATE and _cal_complete(_saved_cal):
    CAL = _saved_cal
    print(f"[reward calibration] skipped: using {CALIBRATION_PATH} (RECALIBRATE = True to redo)")
else:   # no file, RECALIBRATE, or a file from before an arm existed (seeded: the old entries come out the same)
    if _saved_cal is not None and not RECALIBRATE:
        print(f"[reward calibration] {CALIBRATION_PATH} lacks entries or pass settings changed: redoing it")
    if _saved_cal is not None and "ratio" not in _saved_cal:
        for _ext in (".json", "_samples.csv"):
            _old, _keep = CALIBRATION_PATH.replace(".json", _ext), CALIBRATION_PATH.replace(".json", f"_before_ratio{_ext}")
            if os.path.exists(_old) and not os.path.exists(_keep):
                shutil.copy(_old, _keep)
                print(f"[reward calibration] kept the old {os.path.basename(_old)} as {os.path.basename(_keep)}")
    CAL_PROMPTS, TARGET_RATIO, FLOOR_Q, SPREAD_ANCHOR = (16 if SMOKE_RUN else 128), 1.0, 0.01, "correctness"
    G = trainer.args.num_generations
    trainer.vllm_generation.sync_weights()
    cal = trainer.train_dataset.shuffle(seed=1).select(range(min(CAL_PROMPTS, len(trainer.train_dataset))))
    ids = [chat_prompt_ids(tokenizer, p) for p in cal["prompt"]]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(n=G, temperature=trainer.args.temperature, max_tokens=MAX_COMPLETION_LENGTH, seed=0), use_tqdm=True)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    prompts = [p for p in cal["prompt"] for _ in range(G)]
    numbers = [n for n in cal["numbers"] for _ in range(G)]
    targets = [t for t in cal["target"] for _ in range(G)]
    texts = [c.text for o in outs for c in o.outputs]
    completions = [[{"role": "assistant", "content": t}] for t in texts]
    df = pd.DataFrame({"group": np.repeat(np.arange(len(cal)), G), "completion": texts,
                       "correctness": R.correctness_reward_func(prompts, completions, numbers, targets)})
    df["correct"] = df.correctness > 0
    df["format_total"] = sum(np.array(f(completions=completions, numbers=numbers, target=targets), dtype=float)
                             for f in FORMAT_FUNCS)
    df["mean"], df["worst"], df["sum"], df["ref_sum"] = judge_raw(trainer, prompts, completions)
    df["ratio"] = df["sum"] - df["ref_sum"]
    present = df["mean"].notna()
    wrong_scored = present & ~df.correct
    wrong_median = lambda col, fallback: float(df.loc[wrong_scored, col].median()) if wrong_scored.any() else fallback

    def within_std(col, mask):
        return df[mask].groupby("group")[col].std(ddof=0).mean()

    def win(col, mask):
        rate, pairs = within_puzzle_win_rate(df.group[mask].to_numpy(), df.loc[mask, col].to_numpy(), df.correct[mask].to_numpy())
        return rate, pairs

    _all_rows = np.ones(len(df), dtype=bool)
    spreads = {"correctness": within_std("correctness", _all_rows), "format": within_std("format_total", _all_rows)}
    target_spread = TARGET_RATIO * spreads[SPREAD_ANCHOR]
    assert target_spread > 1e-6, f"the {SPREAD_ANCHOR} reward has no spread within groups: {spreads}"
    m = df.loc[present, "mean"]
    floor_m, med_m = float(m.quantile(FLOOR_Q)), float(m.median())
    df["v_mean"] = df["mean"].clip(lower=floor_m)
    cal_mean = {"floor": floor_m, "missing_below_floor": floor_m - 0.05 * (med_m - floor_m),
                "coef": float(target_spread / within_std("v_mean", present))}
    cal_mean["missing_wrong_median"] = wrong_median("v_mean", cal_mean["missing_below_floor"])

    sm = df.loc[present, "sum"]
    floor_sum, med_sum = float(sm.quantile(FLOOR_Q)), float(sm.median())
    df["v_sum"] = df["sum"].clip(lower=floor_sum)
    cal_sum = {"floor": floor_sum, "missing_below_floor": floor_sum - 0.05 * (med_sum - floor_sum),
               "coef": float(target_spread / within_std("v_sum", present))}
    cal_sum["missing_wrong_median"] = wrong_median("v_sum", cal_sum["missing_below_floor"])

    if "ratio" in JUDGE_ARMS:
        assert df.loc[present, "ref_sum"].notna().all(), "ratio: scored completions without a reference score"
        rt = df.loc[present, "ratio"]
        floor_r, ceil_r, med_r = float(rt.quantile(FLOOR_Q)), float(rt.quantile(1 - FLOOR_Q)), float(rt.median())
        df["v_ratio"] = df["ratio"].clip(lower=floor_r, upper=ceil_r)
        cal_ratio = {"floor": floor_r, "ceiling": ceil_r, "missing_below_floor": floor_r - 0.05 * (med_r - floor_r),
                     "coef": float(target_spread / within_std("v_ratio", present))}
        cal_ratio["missing_wrong_median"] = wrong_median("v_ratio", cal_ratio["missing_below_floor"])

    pw = present & df["worst"].notna()
    w = df.loc[pw, "worst"]
    floor_w, med_w = float(w.quantile(FLOOR_Q)), float(w.median())
    df["c_worst"] = df["worst"].clip(lower=floor_w)
    mu_m, sd_m = float(df.loc[present, "v_mean"].mean()), float(df.loc[present, "v_mean"].std())
    mu_w, sd_w = float(df.loc[pw, "c_worst"].mean()), float(df.loc[pw, "c_worst"].std())
    missing_z_worst = (floor_w - mu_w) / sd_w - 0.05 * (med_w - floor_w) / sd_w
    df["v_mean_worst"] = ((df["v_mean"] - mu_m) / sd_m
                          + np.where(df["worst"].isna(), missing_z_worst, (df["c_worst"] - mu_w) / sd_w))
    s = df.loc[present, "v_mean_worst"]
    floor_s, med_s = float(s.quantile(FLOOR_Q)), float(s.median())
    cal_mw = {"floor_mean": floor_m, "mu_mean": mu_m, "sd_mean": sd_m, "floor_worst": floor_w, "mu_worst": mu_w,
              "sd_worst": sd_w, "missing_z_worst": float(missing_z_worst),
              "missing_below_floor": floor_s - 0.05 * (med_s - floor_s),
              "coef": float(target_spread / within_std("v_mean_worst", present))}
    cal_mw["missing_wrong_median"] = wrong_median("v_mean_worst", cal_mw["missing_below_floor"])

    # pass/fail: threshold on the raw mean; the value is 0/1 for every completion (missing = 0)
    thr = (float(PASS_THRESHOLD) if PASS_THRESHOLD is not None
           else float(df.loc[wrong_scored, "mean"].quantile(1 - PASS_MAX_FALSE_PASS)))
    df["v_pass"] = (df["mean"] > thr).astype(float)            # NaN > thr is False: missing = 0
    cal_pass = {"threshold": thr, "threshold_setting": [PASS_THRESHOLD, PASS_MAX_FALSE_PASS],
                "pass_correct": float(df.loc[present & df.correct, "v_pass"].mean()),
                "pass_wrong": float(df.loc[wrong_scored, "v_pass"].mean()),
                "coef": float(target_spread / max(within_std("v_pass", _all_rows), 1e-6))}

    print(f"{len(df)} completions = {len(cal)} puzzles x {G}; accuracy {df.correct.mean():.3f}; "
          f"scored (solution within {PLAUSIBILITY_MAX_SOLUTION_TOKENS} tokens) {present.mean():.3f}; "
          f"with '=' lines {pw.mean():.3f}")
    print(f"within-group spread: correctness {spreads['correctness']:.4f}, format {spreads['format']:.4f} "
          f"(format mean {df.format_total.mean():.3f}); anchor {SPREAD_ANCHOR} x {TARGET_RATIO} = {target_spread:.4f}")
    for name, col, c in ((("mean", "v_mean", cal_mean), ("sum", "v_sum", cal_sum), ("mean_worst", "v_mean_worst", cal_mw))
                         + ((("ratio", "v_ratio", cal_ratio),) if "ratio" in JUDGE_ARMS else ())):
        print(f"  {name:10s}: within-group spread {within_std(col, present):.4f} -> coef {c['coef']:.4g}; missing "
              f"{c['missing_below_floor']:.4g} (below floor) / {c['missing_wrong_median']:.4g} (wrong median)")
    print(f"  pass      : threshold {thr:.3f} on the mean -> passes {cal_pass['pass_correct']:.3f} of scored correct, "
          f"{cal_pass['pass_wrong']:.3f} of scored wrong; within-group spread {within_std('v_pass', _all_rows):.4f} "
          f"-> coef {cal_pass['coef']:.4g}")
    print("judge win rate (correct over wrong, same puzzle) on these groups: "
          + ", ".join(f"{col} {win(col, present & df[col].notna())[0]:.3f} ({win(col, present & df[col].notna())[1]} pairs)"
                      for col in ("mean", "sum", "worst", "v_mean_worst") + (("ratio",) if "ratio" in JUDGE_ARMS else ())))
    CAL = {"mean": cal_mean, "sum": cal_sum, "mean_worst": cal_mw, "pass": cal_pass,
           **({"ratio": cal_ratio} if "ratio" in JUDGE_ARMS else {}), "spread_anchor": SPREAD_ANCHOR, "target_ratio": TARGET_RATIO,
           "spreads": spreads, "cal_prompts": len(cal), "accuracy": float(df.correct.mean()),
           "solution_rate": float(present.mean()), "max_solution_tokens": PLAUSIBILITY_MAX_SOLUTION_TOKENS,
           "adapter": ADAPTER_PATH, "judge": JUDGE_ADAPTER_PATH or "base",
           "policy_start": POLICY_START if JUDGE == "dpo" else "sft", "created": datetime.now().isoformat(timespec="seconds")}
    os.makedirs(GRPO_DIR, exist_ok=True)
    json.dump(CAL, open(CALIBRATION_PATH, "w"), indent=2)
    df.to_csv(CALIBRATION_PATH.replace(".json", "_samples.csv"), index=False)
    print(f"saved {CALIBRATION_PATH}")

# apply to the current trainer (the plausibility reward reads CAL at call time; the weight is set here)
for i, f in enumerate(trainer.reward_funcs):
    if getattr(f, "__name__", "") == "plausibility":
        trainer.reward_weights[i] = plausibility_weight()
print(f"ARM={ARM}: plausibility weight {plausibility_weight():.4g}; "
      + (f"missing = {MISSING_AT} ({CAL[ARM]['missing_' + MISSING_AT]:.4g}); " if ARM in ("mean", "sum", "mean_worst", "ratio") else "")
      + (f"pass threshold {CAL['pass']['threshold']:.3f}; " if ARM == "pass" else "")
      + ("trained" if ARM in JUDGE_ARMS else "logged only (weight 0)"))

In [ ]:
# [early stop] (RFT notebook: run it for CURVE_SET and the report's "start" column; its callback is never used, since
# nothing calls trainer.train()) Run after [held-out] and [reward calibration], before [train]. Every checkpoint is kept, so the best
# can be chosen afterwards by the held-out curve.
#   held-out curve  every HELDOUT_EVAL_STEPS steps: greedy accuracy on CURVE_STRATA (the held-out set from the
#                   candidates + unscored; step 0 = the cached
#                   [held-out] "before" eval). Stop when it dropped in two consecutive evals AND the latest is STOP_DROP
#                   below the best (single dips are noise). Labels are used only for this model selection, never in the
#                   reward (except the correctness arm); same rule for all arms.
#   collapse guards label-free, from the training logs (every 5 steps): more than TRUNC_STOP of completions truncated,
#                   or (continuous judge arms; not "pass") more than ZERO_STD_STOP of groups with zero reward variance,
#                   for GUARD_LOGS logs
#   pass@k          at PASS_EVAL_STEPS: PASS_AT_K samples on the PASS_N puzzles of [held-out] (LOG_DIR/pass_curve.csv)
#   rich evals      at RICH_EVAL_STEPS: the end-of-run evals (test set, held-out strata, mixed 3/4-number test), tagged
#                   step{N} in LOG_DIR (skipped when eval_test_step{N}.json exists: survives a resume)
# The curve is saved to LOG_DIR/heldout_curve.csv and survives a resume.
from datasets import concatenate_datasets

HELDOUT_EVAL_STEPS = 5 if SMOKE_RUN else 25
STOP_DROP = 1.0                  # off for the long validation run (all checkpoints are kept)
TRUNC_STOP, ZERO_STD_STOP, GUARD_LOGS = 0.5, 0.5, 2
CURVE_SET = concatenate_datasets([heldout_dataset(s) for s in CURVE_STRATA])

def curve_eval(trainer):
    trainer.vllm_generation.sync_weights()
    s, _ = eval_puzzles(trainer, CURVE_SET, "curve", quiet=True)
    return {k: float(s[k]) for k in ("accuracy", "accuracy_finished", "truncated", "format_ok", "tokens")}

def start_results():
    """The starting adapter's results for the report's "start" column (all cached)."""
    mb = mixed_before_cached()
    return {"test4": eval_before, "test3": mb["test3"] if mb else None,
            "curve": {**early_stop.curve[0], "n": len(CURVE_SET)}, "pass": pass_before}

_CURVE_KEYS = ("accuracy", "accuracy_finished", "truncated", "format_ok", "tokens")

class EarlyStop(TrainerCallback):
    def __init__(self, trainer, path):
        self.trainer, self.path, self.flags = trainer, path, 0
        if os.path.exists(path):
            self.curve = pd.read_csv(path).to_dict("records")
        else:   # step 0 from the cached "before" held-out eval (same greedy settings, same puzzles)
            b = pd.concat([pd.read_csv(f"{_before_dir}/eval_heldout_{s}_before_samples.csv") for s in CURVE_STRATA
                           if os.path.exists(f"{_before_dir}/eval_heldout_{s}_before_samples.csv")])
            self.curve = [{"step": 0, "accuracy": float(b.correct.mean()),
                           "accuracy_finished": float(b.loc[~b.truncated.astype(bool), "correct"].mean()),
                           "truncated": float(b.truncated.mean()), "format_ok": float(b.format_ok.mean()),
                           "tokens": float(b.tokens.mean())}]

    def _stop(self, control, why):
        print(f"[early stop] step {self.trainer.state.global_step}: {why}")
        control.should_training_stop = True
        control.should_save = True

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        here = {}                                   # this step's results, printed as ONE report below
        _pass_path = f"{LOG_DIR}/pass_curve.csv"
        if PASS_AT_K and step in PASS_EVAL_STEPS and not (os.path.exists(_pass_path)
                                                          and step in set(pd.read_csv(_pass_path).step)):
            self.trainer.vllm_generation.sync_weights()
            s, _ = pass_at_k(self.trainer, PASS_SET, f"step{step}_n{len(PASS_SET)}", PASS_AT_K, out_dir=LOG_DIR,
                             strata=PASS_STRATA)
            new = s.reset_index().rename(columns={"index": "set"}).assign(step=step)
            rows = pd.concat(([pd.read_csv(_pass_path)] if os.path.exists(_pass_path) else []) + [new])
            rows.to_csv(_pass_path, index=False)
            here["pass"] = s
        if step in RICH_EVAL_STEPS and not os.path.exists(f"{LOG_DIR}/eval_test_step{step}.json"):
            self.trainer.vllm_generation.sync_weights()
            s, d = eval_puzzles(self.trainer, _test, f"test_step{step}", out_dir=LOG_DIR, quiet=True)
            h = eval_heldout(self.trainer, f"step{step}", LOG_DIR)
            h.to_csv(f"{LOG_DIR}/heldout_step{step}_summary.csv")
            _, s3, _ = eval_mixed(self.trainer, f"step{step}", out_dir=LOG_DIR, test4=(s, d))   # 4-number eval reused
            here.update(test4=s, test3=s3, curve=curve_of(LOG_DIR, f"step{step}"))
        new_point = not (step % HELDOUT_EVAL_STEPS or any(r["step"] == step for r in self.curve))
        if new_point:   # the curve point: from the rich eval's held-out eval when there was one (same puzzles, greedy)
            c = here.get("curve") or curve_eval(self.trainer)
            self.curve.append({"step": step, **{k: c[k] for k in _CURVE_KEYS}})
            pd.DataFrame(self.curve).to_csv(self.path, index=False)
        if here:
            report(f"step {step}", f"training weights at step {step}", {"start": start_results(), f"step {step}": here},
                   out=f"{LOG_DIR}/report_step{step}.csv")
        if not new_point:
            return
        accs = [r["accuracy"] for r in self.curve]
        print(curve_line(self.curve))
        if len(accs) >= 3 and accs[-1] < accs[-2] < accs[-3] and accs[-1] <= max(accs) - STOP_DROP:
            self._stop(control, f"held-out accuracy fell twice in a row and is {max(accs) - accs[-1]:.3f} below the best")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "reward" not in logs:
            return
        trunc, zero = logs.get("completions/clipped_ratio", 0.0), logs.get("frac_reward_zero_std", 0.0)
        # zero reward variance is collapse only when a continuous reward is trained (saturated format rewards give it
        # by design in the control arm)
        zero_bad = zero > ZERO_STD_STOP and ARM in ("mean", "sum", "mean_worst", "ratio")   # "pass": flat all-wrong groups by design
        self.flags = self.flags + 1 if (trunc > TRUNC_STOP or zero_bad) else 0
        if self.flags >= GUARD_LOGS:
            self._stop(control, f"collapse guard: truncated {trunc:.2f}, zero-variance groups {zero:.2f} "
                                f"({GUARD_LOGS} logs in a row)")

trainer.callback_handler.callbacks = [c for c in trainer.callback_handler.callbacks if type(c).__name__ != "EarlyStop"]
early_stop = EarlyStop(trainer, f"{LOG_DIR}/heldout_curve.csv")
trainer.add_callback(early_stop)
print(f"early stop installed: held-out curve every {HELDOUT_EVAL_STEPS} steps on {len(CURVE_SET)} puzzles "
      f"(step 0 accuracy {early_stop.curve[0]['accuracy']:.3f}); guards truncated > {TRUNC_STOP}, "
      f"zero-variance groups > {ZERO_STD_STOP} for {GUARD_LOGS} logs")

In [ ]:
# [rft] Rejection-sampling fine-tuning rounds (RFT settings in [config]). Per round r: the next RFT_PUZZLES puzzles of
# the training set (seeded shuffle) x NUM_GENERATIONS samples (vLLM, T TEMPERATURE, seed SEED + r) -> scores (format +
# judge or checker) -> per puzzle the top RFT_KEEP finished completions above the group mean -> SFT (policy LoRA, loss
# on completion tokens) -> vLLM synced -> evals as the GRPO notebook's "after" (4- and 3-number test, held-out strata /
# curve puzzles greedy, pass@k sampled; tagged round<r>, the last round "after") and one report table. Saved per round
# in LOG_DIR: round<r>/ (adapter_model.safetensors + adapter_config.json, optimizer.pt, pool.csv: every completion with
# its scores, selected.csv), rft_log.csv (per SFT batch), rounds.csv (per round), report_round<r>.csv.
import time, random, shutil
from peft import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import load_file, save_file
from trl.trainer.utils import selective_log_softmax
from vllm import SamplingParams

assert "grpo_split" in globals() and "CURVE_SET" in globals(), "run [held-out] and [early stop] first"
assert CAL is not None, "run [reward calibration] first (its file is cached for the paper's judges)"
if SELECT == "judge":
    assert plausibility_weight() > 0, f"no calibrated coef for ARM {ARM} in {CALIBRATION_PATH}"
policy = trainer.model
_EOT = tokenizer.convert_tokens_to_ids("<|eot_id|>")
_STOP_IDS = {tokenizer.eos_token_id, _EOT}
_order = train_dataset.shuffle(seed=SEED)          # the rounds' puzzles: consecutive slices of one seeded order
assert len(_order) >= RFT_ROUNDS * RFT_PUZZLES, (len(_order), RFT_ROUNDS * RFT_PUZZLES)
_named = [(n, p) for n, p in policy.named_parameters() if p.requires_grad]
assert _named and all(".default." in n and "lora_" in n for n, _ in _named), "only the policy LoRA may be trainable"
_params = [p for _, p in _named]
opt = torch.optim.AdamW(_params, lr=RFT_LR, betas=(0.9, ADAM_BETA2), weight_decay=WEIGHT_DECAY)
os.makedirs(LOG_DIR, exist_ok=True)
json.dump({**{k: v for k, v in globals().items() if k.isupper() and not k.startswith('_')
              and isinstance(v, (bool, int, float, str, tuple, list, type(None)))}}, open(f"{LOG_DIR}/config.json", "w"),
          indent=1, default=str)
_jl = f"{os.path.dirname(JUDGE_ADAPTER_PATH)}/labels.json" if JUDGE_ADAPTER_PATH else None
json.dump({"selection": SELECT,
           "checker_selection_completions": RFT_ROUNDS * RFT_PUZZLES * NUM_GENERATIONS if SELECT == "checker" else 0,
           "judge": (json.load(open(_jl)) if _jl and os.path.exists(_jl) else "unknown") if SELECT == "judge" else "not used",
           "checker_calibration_completions": 128 * NUM_GENERATIONS if SELECT == "judge" else 0},
          open(f"{LOG_DIR}/labels.json", "w"), indent=1)

def round_dir(r):
    return f"{LOG_DIR}/round{r}"

def save_round(r):
    d = round_dir(r)
    os.makedirs(d, exist_ok=True)
    save_file({k: v.detach().cpu().contiguous() for k, v in get_peft_model_state_dict(policy, adapter_name="default").items()},
              f"{d}/adapter_model.safetensors")
    shutil.copy(f"{_policy_init}/adapter_config.json", f"{d}/adapter_config.json")
    torch.save(opt.state_dict(), f"{d}/optimizer.pt")           # last: its presence marks the round as done

def load_round(r):
    res = set_peft_model_state_dict(policy, load_file(f"{round_dir(r)}/adapter_model.safetensors"), adapter_name="default")
    assert not res.unexpected_keys, res.unexpected_keys[:4]
    opt.load_state_dict(torch.load(f"{round_dir(r)}/optimizer.pt"))

def score_pool(ds, outs):
    """Every completion of the round with its scores: format, checker (measured; trained on only with SELECT 'checker'),
    the judge's raw sum and its plausibility value, and the selection score."""
    rows, prompts, comps, nums, tgts = [], [], [], [], []
    for i, o in enumerate(outs):
        for k, c in enumerate(o.outputs):
            rows.append({"pid": ds[i]["pid"], "puzzle": i, "numbers": json.dumps([int(x) for x in ds[i]["numbers"]]),
                         "target": int(ds[i]["target"]), "sample": k, "text": c.text, "n_tokens": len(c.token_ids),
                         "finished": c.finish_reason == "stop", "_ids": list(c.token_ids)})
            prompts.append(ds[i]["prompt"]); comps.append([{"role": "assistant", "content": c.text}])
            nums.append(ds[i]["numbers"]); tgts.append(ds[i]["target"])
    fmt = sum(np.array(f(completions=comps, numbers=nums, target=tgts), dtype=float) for f in FORMAT_FUNCS)
    corr = np.array(R.correctness_reward_func(prompts, comps, nums, tgts), dtype=float)
    m, w, sm, rs = judge_raw(trainer, prompts, comps)
    judge = np.array(plausibility_value(m, w, sm, ARM if ARM in JUDGE_ARMS else "sum", rs), dtype=float)
    score = fmt + (plausibility_weight() * judge if SELECT == "judge" else corr)
    df = pd.DataFrame(rows)
    df["format"], df["correct"], df["judge_sum"], df["judge_value"], df["score"] = fmt, corr > 0, sm, judge, score
    return df

def select(df):
    """Per puzzle: finished completions above the group mean score, best RFT_KEEP of them (ties: lower sample index);
    then, with RFT_SKIP_Q, only those scored at or above that quantile of the round's selected scores."""
    g = df.groupby("puzzle").score.transform("mean")
    cand = df[df.finished & (df.score > g + 1e-9)]
    sel = cand.sort_values(["puzzle", "score", "sample"], ascending=[True, False, True]).groupby("puzzle").head(RFT_KEEP)
    if RFT_SKIP_Q and len(sel):
        sel = sel[sel.score >= sel.score.quantile(RFT_SKIP_Q)]
    return sel

def sft(ds, sel, r):
    """One SFT pass per epoch over the selected completions; returns the batch log rows."""
    seqs = []
    for _, row in sel.iterrows():
        c = list(row["_ids"])
        if c and c[-1] not in _STOP_IDS:              # finished by a stop token vLLM left out: teach the stop
            c = c + [_EOT]
        seqs.append((chat_prompt_ids(tokenizer, ds[int(row["puzzle"])]["prompt"]), c))
    policy.config.use_cache = False
    policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    policy.train()
    log, rng = [], random.Random(SEED + r)
    for ep in range(RFT_EPOCHS):
        rng.shuffle(seqs)
        for b in range(0, len(seqs), RFT_BATCH):
            batch = seqs[b:b + RFT_BATCH]
            n_tok = sum(len(c) for _, c in batch)
            opt.zero_grad(set_to_none=True)
            tot = 0.0
            for mb in range(0, len(batch), RFT_MICRO):
                chunk = batch[mb:mb + RFT_MICRO]
                L = max(len(p) + len(c) for p, c in chunk)
                ids = torch.full((len(chunk), L), tokenizer.pad_token_id, dtype=torch.long)
                attn, mask = torch.zeros_like(ids), torch.zeros((len(chunk), L - 1), dtype=torch.float32)
                for j, (p, c) in enumerate(chunk):
                    ids[j, :len(p) + len(c)], attn[j, :len(p) + len(c)] = torch.tensor(p + c), 1
                    mask[j, len(p) - 1:len(p) + len(c) - 1] = 1.0          # positions that predict completion tokens
                ids, attn, mask = ids.to(policy.device), attn.to(policy.device), mask.to(policy.device)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = policy(input_ids=ids, attention_mask=attn).logits[:, :-1]
                lp = selective_log_softmax(logits, ids[:, 1:]).float()
                loss = -(lp * mask).sum() / n_tok
                loss.backward()
                tot += float(loss)
                del logits, lp, loss
            gn = float(torch.nn.utils.clip_grad_norm_(_params, RFT_MAX_GRAD_NORM))
            opt.step()
            log.append({"round": r, "epoch": ep, "batch": b // RFT_BATCH, "nll_per_token": tot, "grad_norm": gn,
                        "sequences": len(batch), "tokens": n_tok})
    policy.eval()
    opt.zero_grad(set_to_none=True)
    return log

_done = [r for r in range(1, RFT_ROUNDS + 1) if os.path.exists(f"{round_dir(r)}/optimizer.pt")]
_last = max(_done, default=0)
assert _done == list(range(1, _last + 1)), f"rounds on Drive are not contiguous: {_done}"
if _last:
    load_round(_last)
    print(f"[rft] resuming after round {_last}: policy and optimizer loaded from {round_dir(_last)}")
_rounds_csv, _log_csv = f"{LOG_DIR}/rounds.csv", f"{LOG_DIR}/rft_log.csv"
_reports = {}
for r in range(_last + 1, RFT_ROUNDS + 1):
    t0 = time.time()
    ds = _order.select(range((r - 1) * RFT_PUZZLES, r * RFT_PUZZLES))
    ids = [chat_prompt_ids(tokenizer, p) for p in ds["prompt"]]
    trainer.vllm_generation.sync_weights()
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(n=NUM_GENERATIONS, temperature=TEMPERATURE, max_tokens=MAX_COMPLETION_LENGTH, seed=SEED + r),
        use_tqdm=True)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    t_gen = time.time()
    pool = score_pool(ds, outs)
    sel = select(pool)
    t_score = time.time()
    log = sft(ds, sel, r)
    t_sft = time.time()
    save_round(r)
    pool.drop(columns="_ids").to_csv(f"{round_dir(r)}/pool.csv", index=False)
    sel.drop(columns="_ids").to_csv(f"{round_dir(r)}/selected.csv", index=False)
    pd.concat(([pd.read_csv(_log_csv)] if os.path.exists(_log_csv) else []) + [pd.DataFrame(log)]).to_csv(_log_csv, index=False)

    # evals, as the GRPO notebook's end of training (file tags round<r>; the last round also "after")
    trainer.vllm_generation.sync_weights()
    CURRENT_WEIGHTS = f"{RUN_NAME} after round {r} ({r * RFT_PUZZLES * NUM_GENERATIONS:,} rollouts)"
    tg = "after" if r == RFT_ROUNDS else f"round{r}"
    e4, d4 = eval_puzzles(trainer, _test, f"test_{tg}", out_dir=LOG_DIR, quiet=True)
    _, e3, _ = eval_mixed(trainer, tg, LOG_DIR, test4=(e4, d4))
    eval_heldout(trainer, tg, LOG_DIR)
    pk = (pass_at_k(trainer, PASS_SET, f"{tg}_n{len(PASS_SET)}", PASS_AT_K, out_dir=LOG_DIR, strata=PASS_STRATA)[0]
          if PASS_AT_K else None)
    cv = curve_of(LOG_DIR, tg)
    _reports[r] = {"test4": e4, "test3": e3, "curve": cv, "pass": pk}
    report(f"RFT round {r} of {RFT_ROUNDS}", CURRENT_WEIGHTS, {"start": start_results(), f"round {r}": _reports[r]},
           out=f"{LOG_DIR}/report_round{r}.csv")
    by = pool.groupby("puzzle")
    row = {"round": r, "rollouts": r * RFT_PUZZLES * NUM_GENERATIONS,
           "pool correct": pool.correct.mean(), "pool finished": pool.finished.mean(),
           "puzzles with a selection": sel.puzzle.nunique() / RFT_PUZZLES, "selected": len(sel),
           "selected correct (measured)": sel.correct.mean() if len(sel) else float("nan"),
           "selected tokens": sel.n_tokens.mean() if len(sel) else float("nan"),
           "sft nll/token (last batch)": log[-1]["nll_per_token"] if log else float("nan"),
           "curve accuracy": cv["accuracy"], "test4 accuracy": e4["accuracy"],
           "pass@1": float(pk.loc["total", "pass1"]) if pk is not None else float("nan"),
           f"pass@{PASS_AT_K}": float(pk.loc["total", f"pass{PASS_AT_K}"]) if pk is not None else float("nan"),
           "min generate": (t_gen - t0) / 60, "min score": (t_score - t_gen) / 60, "min sft": (t_sft - t_score) / 60,
           "min total": (time.time() - t0) / 60}
    pd.concat(([pd.read_csv(_rounds_csv)] if os.path.exists(_rounds_csv) else []) + [pd.DataFrame([row])]).to_csv(
        _rounds_csv, index=False)
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
        print(f"\n[rft] rounds so far (pool: {RFT_PUZZLES} training puzzles x {NUM_GENERATIONS} at T {TEMPERATURE:g}; "
              f"curve: held-out curve puzzles, greedy, {len(CURVE_SET)} x 1; test4: 4-number test, greedy, {len(_test)} x 1; "
              f"pass@k: pass@{PASS_AT_K} puzzles, T {PASS_TEMPERATURE:g}, {len(PASS_SET)} x {PASS_AT_K})\n"
              + pd.read_csv(_rounds_csv).to_string(index=False))
print(f"[rft] done: {RFT_ROUNDS} rounds in {LOG_DIR}")


In [ ]:
# [test pass@k] (optional, inference only; after [build], no [rft] needed) Sampled pass@1 / pass@8 / distinct correct
# answers on the full test sets (test_set: 200 four-number, test3_set: 200 three-number) for the RFT rounds in TEST_ROUNDS
# of this run (LOG_DIR/round<r>/), TEST_PASS_K samples per puzzle at PASS_TEMPERATURE (pass_at_k, vLLM seed 0): the
# measurement of GRPO_Countdown_Llama's [test pass@k] (tab_format_vs_corrupt), whose cached start
# (GRPO_DIR/test_pass{k}_before_...) is printed alongside. Loads each round's adapter, then RESTORES the session's weights.
# Cached (a finished round is not sampled again) in LOG_DIR/eval_test_pass{k}_round<r>/ (eval_test4_pass{k}*, eval_test3_pass{k}*).
import math
from peft import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import load_file

TEST_ROUNDS = (4, 8)          # round 4 = the rollouts of 150 GRPO steps (4 x RFT_PUZZLES x NUM_GENERATIONS = 19,200)
TEST_PASS_K = 8               # samples per puzzle, as the GRPO notebook's [test pass@k] and Qwen's grade files

_tp_sets = {"test4": test_set, "test3": test3_set}
_mixed = f"{GRPO_DIR}/eval_mixed_before_n{len(test3_set)}_max{MAX_COMPLETION_LENGTH}"   # the GRPO notebook's start evals
for _s, _ds in _tp_sets.items():          # the same puzzles as the GRPO notebook's (pid, numbers, target)
    _ref = pd.read_csv(f"{_mixed}/eval_{_s}_before_samples.csv")
    assert (list(_ref.pid) == list(_ds["pid"]) and list(_ref.target) == list(_ds["target"])
            and [json.loads(x) for x in _ref.numbers] == [list(n) for n in _ds["numbers"]]), \
        f"[test pass@k] {_s} differs from the GRPO notebook's test puzzles"
_tp_jobs = {f"round{r}": (f"{LOG_DIR}/round{r}/adapter_model.safetensors", f"{LOG_DIR}/eval_test_pass{TEST_PASS_K}_round{r}")
            for r in TEST_ROUNDS}
_start_dir = f"{GRPO_DIR}/test_pass{TEST_PASS_K}_before_n{len(test_set)}+{len(test3_set)}_max{MAX_COMPLETION_LENGTH}"
_missing = [a for a, _ in _tp_jobs.values() if not os.path.exists(a)]
assert not _missing, f"[test pass@k] no adapter: {_missing}"
_tp_file = lambda d, s: f"{d}/eval_{s}_pass{TEST_PASS_K}_samples.csv"
_tp_done = lambda d: all(os.path.exists(_tp_file(d, s)) for s in _tp_sets)

_todo = {n: j for n, j in _tp_jobs.items() if not _tp_done(j[1])}
print(f"[test pass@k] to sample: {list(_todo) or 'nothing (all cached)'}")
if _todo:
    _now = {k: v.detach().clone() for k, v in get_peft_model_state_dict(trainer.model, adapter_name="default").items()}
    try:
        for _name, (_adapter, _out) in _todo.items():
            _res = set_peft_model_state_dict(trainer.model, load_file(_adapter), adapter_name="default")
            assert not _res.unexpected_keys, _res.unexpected_keys[:4]
            trainer.vllm_generation.sync_weights()
            for _s, _ds in _tp_sets.items():
                pass_at_k(trainer, _ds, _s, TEST_PASS_K, out_dir=_out)
            print(f"[test pass@k] {_name} -> {_out}")
    finally:
        set_peft_model_state_dict(trainer.model, _now, adapter_name="default")
        trainer.vllm_generation.sync_weights()

_rows = []
for _name, _out in [("start", _start_dir)] + [(n, o) for n, (_, o) in _tp_jobs.items()]:
    if not _tp_done(_out):
        print(f"[test pass@k] {_name}: no results in {_out} (the start comes from GRPO_Countdown_Llama [test pass@k])")
        continue
    _d = {s: pd.read_csv(_tp_file(_out, s)) for s in _tp_sets}
    _d["test3+4"] = pd.concat(_d.values())
    for _s, d in _d.items():
        _rows.append({"weights": _name, "set": _s, "puzzles": len(d), "pass@1": d.pass1.mean(),
                      f"pass@{TEST_PASS_K}": d[f"pass{TEST_PASS_K}"].mean(), "distinct correct": d.distinct_correct.mean(),
                      "truncated": d.truncated.mean()})
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(f"\n[test pass@k] T {PASS_TEMPERATURE:g}, {TEST_PASS_K} samples per puzzle; RFT run {RUN_NAME}")
    print(pd.DataFrame(_rows).set_index(["set", "weights"]).sort_index().to_string())